# SEC-C GNN V4 Training Notebook
## MiniGINv3 + APS Conformal Prediction for Vulnerability Detection

**Framework role:** Trains Module 2 (Graph stage) of the SEC-C 4-stage cascade.
The model produces `structural_risk_score` (β-weight = 30% in fusion) and APS
conformal prediction sets that gate escalation from Stage 2 → Stage 3 (LLM).

---

### V4 Fixes Over V3

| Problem (V3) | Fix (V4) |
|---|---|
| `max_per_language=3000` discarded 20K C/C++ samples | Raised to **20,000** — uses full ~23K available |
| 3,032 total samples → 0.6 min training | **~20,000 samples** → 35–50 min training |
| Conformal threshold=1.0, 99.7% ambiguous | More data → better generalisation → real threshold |
| CrossVul: all 3 HF IDs failed | **`hitoshura25/crossvul`** (9.3K multi-lang pairs, correct ID) |
| CVEfixes: only 11 Python vuln samples | **`DetectVul/CVEFixes`** (5.7K Python stmts, fixed ID) |
| Juliet C/C++: 150 samples (HF truncated) | **`LorenzH/juliet_test_suite_c_1_3`** (101K available) |
| No PrimeVul | **`starsofchance/PrimeVul`** (236K deduplicated, hardest benchmark) |
| No Python statement-level data | **`DetectVul/Vudenc`** (15.8K Python CVE stmts) |
| `alpha=0.2` (80% coverage) | **`alpha=0.1`** (90% coverage, aligned with framework config) |
| `patience=25` | **`patience=20`** |
| `dropout=0.4` | **`dropout=0.35`** (less regularisation with 8x more data) |

### V3 Fixes (inherited, kept in V4)

| Problem (V2) | Fix (retained) |
|---|---|
| Focal(gamma=2) + inverse weights → predict-all-vulnerable | Standard CE + label_smoothing=0.1 + mild class_weight=[1, 1.5] |
| LR=1e-3, best at epoch 3, 1.1 min training | LR=3e-4, 5-epoch warmup, cosine decay |
| Early stopping on val_loss (wrong metric) | Early stopping on **val F1** |
| Per-language class imbalance unchecked | Strict 1:1 per-language balancing |
| 2-layer GAT, 298K params, shallow | 3-layer GIN + residual + BatchNorm + dual pooling, 2.37M params |
| Regex-only code graphs | tree-sitter real AST + regex CFG/DDG fallback |

### Expected V4 Results

| Metric | V2 (actual) | V3 (actual) | V4 (target) |
|---|---|---|---|
| Training samples | ~3K (buggy) | 1,819 | **~13,800** |
| Test F1 | 0.57 (degenerate) | 0.652 | **0.68 – 0.74** |
| Test Precision | 0.40 | 0.544 | **0.63 – 0.72** |
| AUC-ROC | — | 0.623 | **0.70 – 0.76** |
| Conformal threshold | 1.0 | 1.0 | **0.78 – 0.92** |
| Conformal singleton rate | 0% | 0.2% | **20 – 40%** |
| Training time | 1.2 min | 0.6 min | **35 – 50 min** |

### Architecture: MiniGINv3

Graph Isomorphism Network (Xu et al., ICLR 2019) — maximally expressive classical
GNN, equivalent to the Weisfeiler-Lehman graph isomorphism test.

```
Input: 774-dim per node  (768 GraphCodeBERT mean-pool + 6 structural features)
  └─ Linear projection  774 → 384
  └─ 3x GINConv(MLP: 384→768→384) + BatchNorm + ReLU + Dropout(0.35) + Residual
  └─ Dual pooling: global_mean_pool + global_add_pool → concat → 768-dim
  └─ Classifier head: Linear(768 → 384 → 2)
  └─ Confidence head: Linear(768 → 1) + Sigmoid
  Total params: 2,375,046
```

**Novel contribution:** First application of APS conformal prediction to
vulnerability detection (Angelopoulos & Bates, ICLR 2024). Provides
distribution-free 90% coverage guarantee, replacing arbitrary thresholds
with principled uncertainty-driven escalation decisions.


## Cell 1: Setup & Install

In [1]:
# ============================================================================
# Cell 1: Setup & Install
# ============================================================================
import subprocess, sys, os

def run_cmd(cmd, label=""):
    print(f"$ {label or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip()[-400:])
    if r.returncode != 0 and r.stderr.strip():
        print(f"  WARN: {r.stderr.strip()[-200:]}")
    return r.returncode

import torch
tv = torch.__version__.split('+')[0]
ctag = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"PyTorch: {tv}, CUDA: {ctag}")

# PyG
pyg_url = f"https://data.pyg.org/whl/torch-{tv}+cu{ctag}.html"
run_cmd(f"{sys.executable} -m pip install -q torch-scatter torch-sparse torch-cluster -f {pyg_url}", "install pyg deps")
run_cmd(f"{sys.executable} -m pip install -q torch-geometric", "install torch-geometric")
run_cmd(f"{sys.executable} -m pip install -q transformers datasets scikit-learn matplotlib networkx tqdm", "install utils")

# tree-sitter (real AST parsing)
try:
    run_cmd(f"{sys.executable} -m pip install -q 'tree-sitter>=0.22' tree-sitter-python tree-sitter-javascript tree-sitter-java tree-sitter-c tree-sitter-go", "install tree-sitter")
    import tree_sitter_python, tree_sitter_javascript, tree_sitter_java, tree_sitter_c, tree_sitter_go
    from tree_sitter import Language, Parser
    _TS_LANGS = {
        "python":     Language(tree_sitter_python.language()),
        "javascript": Language(tree_sitter_javascript.language()),
        "java":       Language(tree_sitter_java.language()),
        "c_cpp":      Language(tree_sitter_c.language()),
        "go":         Language(tree_sitter_go.language()),
    }
    USE_TREE_SITTER = True
    print("  tree-sitter: OK")
except Exception as _ts_err:
    USE_TREE_SITTER = False
    _TS_LANGS = {}
    print(f"  tree-sitter: unavailable ({_ts_err}) — using regex fallback")

# GPU check
print("\n" + "="*60)
try:
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        gpu_mem = props.total_memory / 1e9
        print(f"  GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB)")
    else:
        print("  WARNING: No GPU — training will be very slow.")
except Exception as e:
    print(f"  GPU info unavailable: {e}")
print("="*60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")


PyTorch: 2.10.0, CUDA: 128
$ install pyg deps
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 115.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 104.9 MB/s eta 0:00:00
$ install torch-geometric
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.4 MB/s eta 0:00:00
$ install utils
$ install tree-sitter
 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.1/98.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.1 MB/s eta 0:00:00
  tree-sitter: OK

  GPU: Tesla T4 (15.6 GB)
  Device: cuda


## Cell 2: Configuration — V4

All hyperparameters in one place. V4 changes from V3:

- `max_per_language = 20000` — **critical fix** (was 3000; was silently discarding 20K real C/C++ samples)
- `alpha = 0.1` — 90% APS coverage guarantee (was 0.2, misaligned with framework `default.yaml`)
- `dropout = 0.35` — less regularisation needed with 8x more training data
- `patience = 20` — tighter early stopping (was 25)
- New per-source caps: `max_primevul=8000`, `max_juliet_c=6000`, `max_crossvul=6000`, `max_vudenc=3000`


In [2]:
# ============================================================================
# Cell 2: Configuration — V5 (all hyperparameters in one place)
# ============================================================================
import torch
from pathlib import Path
import json, random
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CONFIG = {
    # ── Model architecture ──────────────────────────────────────────────────
    "embedding_model":     "microsoft/graphcodebert-base",
    "embedding_dim":       768,
    "node_feature_dim":    6,
    "input_dim":           774,     # 768 GCB mean-pool + 6 structural features
    "hidden_dim":          384,
    "num_gin_layers":      3,
    "dropout":             0.35,    # V4: 0.35 (less regularisation with more data)
    "num_classes":         2,

    # ── Graph construction ──────────────────────────────────────────────────
    "max_nodes":               300,
    "max_tokens_per_node":     128,
    "embedding_batch_size":    64,

    # ── Per-language data caps ──────────────────────────────────────────────
    "target_balance_ratio":    1.0,   # strict 1:1 vuln:safe per language
    "max_per_language":        20000, # V4 FIX: was 3000 (was discarding 20K C/C++ samples)
    "min_per_language":        50,

    # ── Per-source dataset caps ─────────────────────────────────────────────
    "max_bigvul":      10000,
    "max_diversevul":  8000,
    "max_devign":      5000,
    "max_primevul":    8000,   # NEW: starsofchance/PrimeVul
    "max_juliet_c":    6000,   # NEW: LorenzH/juliet_test_suite_c_1_3  (101K avail)
    "max_crossvul":    6000,   # NEW: hitoshura25/crossvul (was failing)
    "max_vudenc":      3000,   # NEW: DetectVul/Vudenc  (Python, 15.8K stmts)
    "max_cvefixes":    3000,   # DetectVul/CVEFixes  (Python stmts)

    # ── Training ────────────────────────────────────────────────────────────
    "batch_size":          64,
    "epochs":              100,
    "lr":                  3e-4,
    "lr_warmup_epochs":    5,
    "weight_decay":        1e-3,
    "patience":            20,    # V4: 20 (was 25)
    "label_smoothing":     0.0,    # V5: was 0.1 (killed conformal singletons)
    "grad_clip":           0.5,
    "class_weight_vuln":   1.5,   # mild; no focal loss

    # ── Conformal prediction ────────────────────────────────────────────────
    "alpha":                   0.1,  # V4 FIX: 0.1 = 90% coverage (was 0.2, misaligned with framework)
    "threshold_search_steps":  81,

    # ── Dataset split ───────────────────────────────────────────────────────
    "train_ratio": 0.60,
    "val_ratio":   0.15,
    "cal_ratio":   0.15,
    "test_ratio":  0.10,

    # ── Language encoding ───────────────────────────────────────────────────
    "language_ids": {
        "python":     0.0,
        "javascript": 0.2,
        "java":       0.4,
        "c_cpp":      0.6,
        "go":         0.8,
    },
}

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"  Architecture: MiniGINv3, {CONFIG['num_gin_layers']}-layer GIN, hidden={CONFIG['hidden_dim']}")
print(f"  Training: LR={CONFIG['lr']}, warmup={CONFIG['lr_warmup_epochs']}ep, patience={CONFIG['patience']}")
print(f"  Balance: {CONFIG['target_balance_ratio']}:1 (strict per-language)")
print(f"  Loss: CE + label_smoothing={CONFIG['label_smoothing']} (no focal loss)")
print(f"  Conformal: alpha={CONFIG['alpha']} ({(1-CONFIG['alpha'])*100:.0f}% coverage target)")
print(f"  Max per language: {CONFIG['max_per_language']} (V4: uncapped for C/C++)")

# ── Sink / Source patterns (defined here so all downstream cells have them) ──
SINK_PATTERNS = [
    "execute", "exec", "system", "popen", "runtime", "processbuilder",
    "sendredirect", "forward", "include", "write", "print", "println",
    "printf", "sprintf", "fprintf", "strcpy", "strcat", "memcpy", "gets",
    "scanf", "fscanf", "sscanf", "fread", "recv", "malloc", "calloc",
    "realloc", "free", "fopen", "open", "connect", "bind", "listen",
    "accept", "eval", "innerhtml", "document.write", "setattribute",
    "preparestatement", "createquery", "executequery", "executeupdate",
    "subprocess", "os.system", "os.popen", "pickle.loads", "yaml.load",
    "cursor.execute", "render_template_string", "child_process",
    "vm.runinnewcontext", "res.send", "res.write", "exec.command",
    "db.query", "db.exec", "fmt.fprintf",
]
SOURCE_PATTERNS = [
    "request", "getparameter", "getquerystring", "getheader",
    "getinputstream", "getreader", "getcookies", "getpathinfo",
    "input", "argv", "stdin", "environ", "getenv", "args",
    "fgets", "fread", "recv", "recvfrom", "read", "readline",
    "readlines", "scanner", "bufferedreader", "fileinputstream",
    "socket", "urlconnection", "httpservletrequest",
    "flask.request", "django.request", "sys.argv", "os.environ",
    "req.body", "req.params", "req.query", "process.env",
    "process.argv", "r.formvalue", "r.url.query", "os.args", "r.body",
]


Configuration loaded.
  Architecture: MiniGINv3, 3-layer GIN, hidden=384
  Training: LR=0.0003, warmup=5ep, patience=20
  Balance: 1.0:1 (strict per-language)
  Loss: CE + label_smoothing=0.0 (no focal loss)
  Conformal: alpha=0.1 (90% coverage target)
  Max per language: 20000 (V4: uncapped for C/C++)


## Cell 3: Multi-Source Dataset Loading — V4

Eight sources with **verified** HuggingFace IDs (3 new, 2 corrected vs V3):

| # | Dataset | HF ID | Lang | Type | Samples |
|---|---------|-------|------|------|---------|
| 1 | BigVul | `bstee615/bigvul` | C/C++ | Real CVE | 217K (cap 10K) |
| 2 | DiverseVul | `claudios/DiverseVul` | C/C++ | Real CVE | 330K (cap 8K) |
| 3 | Devign | `google/code_x_glue_cc_defect_detection` | C/C++ | Real | 27K (cap 5K) |
| 4 | **PrimeVul** *(new)* | `starsofchance/PrimeVul` | C/C++ | Real CVE | 236K dedup (cap 8K) |
| 5 | **Juliet C/C++** *(fixed)* | `LorenzH/juliet_test_suite_c_1_3` | C/C++ | Synthetic NIST | 101K (cap 6K) |
| 6 | **CrossVul** *(fixed ID)* | `hitoshura25/crossvul` | Multi-lang | Real CVE | 9.3K pairs (cap 6K) |
| 7 | **VUDENC** *(new)* | `DetectVul/Vudenc` | Python | Real CVE | 15.8K stmts (cap 3K) |
| 8 | **CVEfixes** *(fixed ID)* | `DetectVul/CVEFixes` | Python | Real CVE | 5.7K stmts (cap 3K) |

All wrapped in `try/except` — any failed HF download is skipped gracefully.

**Column parsing notes:**
- BigVul: `func_before` (vuln) / `func_after` (safe), `vul` label, `CWE ID` column
- PrimeVul: `func` column, `vul` binary label, `CWE ID` column
- CrossVul: `vulnerable_code` → label=1, `fixed_code` → label=0, `cwe_id`, `language`
- Juliet: `bad` → label=1, `good` → label=0, `class` (CWE number as int)
- VUDENC / CVEfixes: `raw_lines` (list of strs), `label` (list of ints per-statement); join lines, label=1 if any stmt vulnerable


In [3]:
# ============================================================================
# Cell 3: Dataset Loading — V4 (verified HuggingFace IDs, corrected schemas)
#
# Sources:
#   C/C++ real:      bstee615/bigvul | claudios/DiverseVul | google/code_x_glue_cc_defect_detection
#   C/C++ quality:   starsofchance/PrimeVul  (deduplicated, hard benchmark)
#   C/C++ synthetic: LorenzH/juliet_test_suite_c_1_3  (101K avail, was 150)
#   Multi-lang:      hitoshura25/crossvul  (was failing with wrong IDs)
#   Python:          DetectVul/Vudenc | DetectVul/CVEFixes
# ============================================================================
import hashlib, re, time, os
from collections import Counter, defaultdict
from datasets import load_dataset

all_samples = []

# ── Helpers ─────────────────────────────────────────────────────────────────
def make_sample(code, label, cwe, lang, source):
    """Return a sample dict or None if code is unusable."""
    if not code or not isinstance(code, str):
        return None
    code = code.strip()
    if len(code) < 20:          # too short to build a meaningful graph
        return None
    return {
        "code":     code,
        "label":    int(label),
        "cwe":      str(cwe).strip() if cwe else "unknown",
        "language": lang,
        "source":   source,
    }

def collect(name, samples, max_count=None):
    """Apply cap, extend all_samples, print stats."""
    if not samples:
        print(f"  [{name}] 0 samples")
        return
    capped = samples[:max_count] if max_count and len(samples) > max_count else samples
    lc = Counter(s["label"] for s in capped)
    lnc = Counter(s["language"] for s in capped)
    all_samples.extend(capped)
    print(f"  [{name}] {len(capped)} samples — {dict(lc)} — langs={dict(lnc)}")

# ── 1. BigVul ────────────────────────────────────────────────────────────────
print("=" * 60)
print("  1. BigVul (C/C++, ~217K real CVE-labeled functions)")
try:
    _bv = load_dataset("bstee615/bigvul", split="train")
    _bv_s = []
    for row in _bv:
        code  = row.get("func_before") or row.get("func_after") or ""
        label = int(row.get("vul", 0))
        cwe   = str(row.get("CWE ID", "unknown") or "unknown")
        _bv_s.append(make_sample(code, label, cwe, "c_cpp", "BigVul"))
    collect("BigVul", [s for s in _bv_s if s], CONFIG["max_bigvul"])
except Exception as _e:
    print(f"  BigVul FAILED: {_e}")

# ── 2. DiverseVul ────────────────────────────────────────────────────────────
print("=" * 60)
print("  2. DiverseVul (C/C++, 330K real CVE-labeled)")
try:
    _dv = load_dataset("claudios/DiverseVul", split="test")
    _dv_s = []
    for row in _dv:
        code  = row.get("func", "") or ""
        label = int(row.get("target", 0))
        cwe_r = row.get("cwe", [])
        cwe   = cwe_r[0] if isinstance(cwe_r, list) and cwe_r else str(cwe_r or "unknown")
        _dv_s.append(make_sample(code, label, cwe, "c_cpp", "DiverseVul"))
    collect("DiverseVul", [s for s in _dv_s if s], CONFIG["max_diversevul"])
except Exception as _e:
    print(f"  DiverseVul FAILED: {_e}")

# ── 3. Devign / CodeXGLUE ────────────────────────────────────────────────────
print("=" * 60)
print("  3. Devign/CodeXGLUE (C/C++, 27K real samples, all splits)")
try:
    _dg_s = []
    for _split in ["train", "validation", "test"]:
        try:
            _dg = load_dataset("google/code_x_glue_cc_defect_detection",
                               split=_split)
            for row in _dg:
                code  = row.get("func", "") or ""
                label = int(row.get("target", 0))
                _dg_s.append(make_sample(code, label, "unknown", "c_cpp", "Devign"))
        except Exception as _e2:
            print(f"    Devign split={_split}: {_e2}")
    collect("Devign", [s for s in _dg_s if s], CONFIG["max_devign"])
except Exception as _e:
    print(f"  Devign FAILED: {_e}")

# ── 4. PrimeVul — V5 FIX: load JSONL directly (HF split metadata broken) ────
print("=" * 60)
print("  4. PrimeVul (C/C++, 236K deduplicated, hardest benchmark)")
try:
    _pv_s = []
    _pv_files = {
        "train": "hf://datasets/starsofchance/PrimeVul/primevul_train.jsonl",
        "valid": "hf://datasets/starsofchance/PrimeVul/primevul_valid.jsonl",
        "test":  "hf://datasets/starsofchance/PrimeVul/primevul_test.jsonl",
    }
    for _split_name, _pv_path in _pv_files.items():
        try:
            _pv = load_dataset("json", data_files=_pv_path, split="train")
            for row in _pv:
                code  = row.get("func", "") or ""
                label = int(row.get("vul", 0))
                cwe   = str(row.get("CWE ID", "unknown") or "unknown")
                _pv_s.append(make_sample(code, label, cwe, "c_cpp", "PrimeVul"))
        except Exception as _e2:
            print(f"    PrimeVul {_split_name}: {_e2}")
    collect("PrimeVul", [s for s in _pv_s if s], CONFIG["max_primevul"])
except Exception as _e:
    print(f"  PrimeVul FAILED: {_e}")

# ── 5. Juliet C/C++ — NEW large source ───────────────────────────────────────
print("=" * 60)
print("  5. Juliet C/C++ (LorenzH/juliet_test_suite_c_1_3, 101K avail)")
try:
    _jc_s = []
    for _split in ["train", "test"]:
        try:
            _jc = load_dataset("LorenzH/juliet_test_suite_c_1_3",
                               split=_split)
            for row in _jc:
                bad_code  = row.get("bad", "") or ""
                good_code = row.get("good", "") or ""
                cwe_num   = str(row.get("class", "") or "")
                cwe       = f"CWE-{cwe_num}" if cwe_num.isdigit() else cwe_num or "unknown"
                _jc_s.append(make_sample(bad_code,  1, cwe, "c_cpp", "Juliet-C"))
                _jc_s.append(make_sample(good_code, 0, cwe, "c_cpp", "Juliet-C"))
        except Exception as _e2:
            print(f"    Juliet-C split={_split}: {_e2}")
    collect("Juliet-C", [s for s in _jc_s if s], CONFIG["max_juliet_c"])
except Exception as _e:
    print(f"  Juliet-C FAILED: {_e}")

# ── 6. CrossVul — FIXED HF ID ────────────────────────────────────────────────
print("=" * 60)
print("  6. CrossVul multi-language (hitoshura25/crossvul, 9.3K pairs)")
try:
    _cv = load_dataset("hitoshura25/crossvul", split="train")
    _cv_s = []
    _lang_map = {
        "c": "c_cpp", "cpp": "c_cpp", "c++": "c_cpp", "c/c++": "c_cpp",
        "python": "python", "py": "python",
        "java": "java",
        "javascript": "javascript", "js": "javascript", "typescript": "javascript",
        "go": "go",
    }
    for row in _cv:
        vuln_code = row.get("vulnerable_code", "") or ""
        safe_code = row.get("fixed_code", "") or ""
        cwe       = str(row.get("cwe_id", "unknown") or "unknown")
        lang_raw  = str(row.get("language", "") or "").lower().strip()
        lang      = _lang_map.get(lang_raw)
        if lang is None:
            continue
        _cv_s.append(make_sample(vuln_code, 1, cwe, lang, "CrossVul"))
        _cv_s.append(make_sample(safe_code, 0, cwe, lang, "CrossVul"))
    collect("CrossVul", [s for s in _cv_s if s], CONFIG["max_crossvul"])
except Exception as _e:
    print(f"  CrossVul FAILED: {_e}")

# ── 7. VUDENC Python — NEW correct ID ────────────────────────────────────────
# Schema: lines/raw_lines/label/type are all LISTS (per-statement).
# We join raw_lines into a function string and derive a binary label:
#   label = 1 if ANY statement in the function is vulnerable, else 0.
print("=" * 60)
print("  7. VUDENC Python (DetectVul/Vudenc, 15.8K statement-level CVE)")
try:
    _vu_s = []
    for _split in ["train", "test"]:
        try:
            _vu = load_dataset("DetectVul/Vudenc", split=_split)
            for row in _vu:
                raw = row.get("raw_lines", []) or row.get("lines", []) or []
                labels = row.get("label", []) or []
                if not isinstance(raw, list) or len(raw) < 3:
                    continue
                code = chr(10).join(raw)
                label = 1 if any(l == 1 for l in labels) else 0
                _vu_s.append(make_sample(code, label, "unknown", "python", "VUDENC"))
        except Exception as _e2:
            print(f"    VUDENC split={_split}: {_e2}")
    collect("VUDENC", [s for s in _vu_s if s], CONFIG["max_vudenc"])
except Exception as _e:
    print(f"  VUDENC FAILED: {_e}")

# ── 8. CVEfixes Python — FIXED ID ────────────────────────────────────────────
# Schema: same as VUDENC — lines/raw_lines/label/type are all LISTS (per-statement).
print("=" * 60)
print("  8. CVEfixes Python (DetectVul/CVEFixes, 5.7K statement-level)")
try:
    _cf_s = []
    for _split in ["train", "test"]:
        try:
            _cf = load_dataset("DetectVul/CVEFixes", split=_split)
            for row in _cf:
                raw = row.get("raw_lines", []) or row.get("lines", []) or []
                labels = row.get("label", []) or []
                if not isinstance(raw, list) or len(raw) < 3:
                    continue
                code = chr(10).join(raw)
                label = 1 if any(l == 1 for l in labels) else 0
                _cf_s.append(make_sample(code, label, "unknown", "python", "CVEfixes-Py"))
        except Exception as _e2:
            print(f"    CVEFixes split={_split}: {_e2}")
    collect("CVEfixes-Py", [s for s in _cf_s if s], CONFIG["max_cvefixes"])
except Exception as _e:
    print(f"  CVEfixes-Py FAILED: {_e}")

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\nRaw total: {len(all_samples)} samples")
_lbl_c = Counter(s["label"] for s in all_samples)
_lng_c = Counter(s["language"] for s in all_samples)
_cwe_k = sum(1 for s in all_samples if s["cwe"] != "unknown")
print(f"  Labels:    vuln={_lbl_c[1]}, safe={_lbl_c[0]}")
print(f"  CWE known: {_cwe_k}/{len(all_samples)} ({100*_cwe_k/max(len(all_samples),1):.1f}%)")
for _lang, _n in sorted(_lng_c.items()):
    _vn = sum(1 for s in all_samples if s["language"]==_lang and s["label"]==1)
    _sn = sum(1 for s in all_samples if s["language"]==_lang and s["label"]==0)
    print(f"  {_lang:<20s} {_n:5d} (v={_vn}, s={_sn})")

_src_c = Counter(s["source"] for s in all_samples)
print("\nSource breakdown:")
for _src, _n in _src_c.most_common():
    print(f"  {_src:<25s} {_n:5d}")


  1. BigVul (C/C++, ~217K real CVE-labeled functions)


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-c6410a8bb202ca(…):   0%|          | 0.00/177M [00:00<?, ?B/s]

data/validation-00000-of-00001-d21ad3921(…):   0%|          | 0.00/37.9M [00:00<?, ?B/s]

data/test-00000-of-00001-d20b0e7149fa6ee(…):   0%|          | 0.00/37.9M [00:00<?, ?B/s]

...

data/train-00000-of-00001-e16c38c4453519(…):   0%|          | 0.00/2.23M [00:00<?, ?B/s]

data/test-00000-of-00001-4be7d978ce504ad(…):   0%|          | 0.00/553k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4584 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1146 [00:00<?, ? examples/s]

  [CVEfixes-Py] 3000 samples — {0: 2964, 1: 36} — langs={'python': 3000}

Raw total: 41000 samples
  Labels:    vuln=17048, safe=23952
  CWE known: 26558/41000 (64.8%)
  c_cpp                34020 (v=16438, s=17582)
  go                     108 (v=54, s=54)
  java                   208 (v=104, s=104)
  javascript             260 (v=130, s=130)
  python                6404 (v=322, s=6082)

Source breakdown:
  BigVul                    10000
  DiverseVul                 8000
  Juliet-C                   6000
  CrossVul                   6000
  Devign                     5000
  VUDENC                     3000
  CVEfixes-Py                3000


## Cell 4: Strict 1:1 Per-Language Balancing — V4

**V4 key fix:** `max_per_language=20000` (was 3000) allows C/C++ to contribute its
full ~15K balanced samples. In V3 this cap threw away 20K samples, causing 0.6-min
training and model overfit on 1,819 samples.

Procedure:
1. **Dedup** by SHA-256 of code string — removes exact duplicates across sources
2. **Length filter** — drop samples shorter than 20 characters
3. **Per-language 1:1 balance** — subsample majority class to match minority
4. **Language gate** — skip languages with < `min_per_language=50` total samples

Expected V4 output: ~15,000 C/C++ + ~3,000 Python + ~300 Java = ~18,300 total


In [4]:
# ============================================================================
# Cell 4: Strict Per-Language 1:1 Balance
# V2 bug: Java was 5.9:1 safe:vuln; balancer only capped over-representation.
# V3 fix: oversample minority to achieve strict 1:1 within each language.
# ============================================================================
import hashlib
from collections import Counter, defaultdict

print("="*60)
print("  V3 Strict 1:1 Per-Language Balancing")
print("="*60)

# 1. Deduplicate by content hash
seen_hashes = set()
deduped = []
for s in all_samples:
    h = hashlib.md5(s["code"].encode("utf-8", errors="ignore")).hexdigest()
    if h not in seen_hashes:
        seen_hashes.add(h)
        deduped.append(s)
print(f"\n1. Dedup: {len(all_samples)} -> {len(deduped)}")
all_samples = deduped

# 2. Filter too-short samples
all_samples = [s for s in all_samples if len(s["code"].strip().split("\n")) >= 3]
print(f"2. Length filter: {len(all_samples)} samples remain")

# 3. Strict 1:1 balance per language
by_lang_label = defaultdict(list)
for s in all_samples:
    by_lang_label[(s["language"], s["label"])].append(s)

balanced = []
print("\n3. Per-language 1:1 balance:")
print(f"   {'Lang':<15} {'Vuln':>6} {'Safe':>6} {'Total':>6}  Action")
print(f"   {'-'*55}")

for lang in CONFIG["language_ids"]:
    vuln_s = by_lang_label[(lang, 1)][:]
    safe_s = by_lang_label[(lang, 0)][:]

    if not vuln_s and not safe_s:
        print(f"   {lang:<15} {'N/A':>6}  -- skipped")
        continue

    max_each = min(
        max(len(vuln_s), len(safe_s)),  # don't shrink more than necessary
        CONFIG["max_per_language"] // 2,
    )
    # Ensure minimum
    if max_each < CONFIG["min_per_language"] // 2:
        max_each = CONFIG["min_per_language"] // 2

    # Actual target: min of available in each class (1:1 strict)
    target = min(max_each, max(len(vuln_s), 1), max(len(safe_s), 1))
    # Ensure both sides have at least min
    target = max(target, min(CONFIG["min_per_language"] // 2, len(vuln_s), len(safe_s)) if (vuln_s and safe_s) else 0)

    if not vuln_s or not safe_s:
        # One side completely missing — skip
        print(f"   {lang:<15} {len(vuln_s):>6} {len(safe_s):>6}  -- one class missing, skip")
        continue

    # Oversample minority / undersample majority
    target = min(target, CONFIG["max_per_language"] // 2)

    if len(vuln_s) < target:
        vuln_bal = random.choices(vuln_s, k=target)   # oversample with replacement
        action_v = f"over({len(vuln_s)}->{target})"
    else:
        vuln_bal = random.sample(vuln_s, target)
        action_v = f"under({len(vuln_s)}->{target})" if len(vuln_s) > target else "ok"

    if len(safe_s) < target:
        safe_bal = random.choices(safe_s, k=target)
        action_s = f"over({len(safe_s)}->{target})"
    else:
        safe_bal = random.sample(safe_s, target)
        action_s = f"under({len(safe_s)}->{target})" if len(safe_s) > target else "ok"

    balanced.extend(vuln_bal + safe_bal)
    print(f"   {lang:<15} {target:>6} {target:>6} {target*2:>6}  v:{action_v} s:{action_s}")

random.shuffle(balanced)
all_samples = balanced

total_v = sum(1 for s in all_samples if s["label"]==1)
total_s = sum(1 for s in all_samples if s["label"]==0)
print(f"\n   Final: {len(all_samples)} samples — vuln={total_v}, safe={total_s}, ratio={total_v/max(total_s,1):.2f}:1")

# CWE coverage after balance
cwe_ctr = Counter(s.get("cwe","?") for s in all_samples)
known = len(all_samples) - cwe_ctr.get("CWE-unknown", 0)
print(f"   CWE labels known: {known}/{len(all_samples)} ({known/max(len(all_samples),1):.1%})")

# Save balanced raw samples checkpoint
import json as _j
_raw_ckpt = OUTPUT_DIR / "raw_samples_v3.json"
try:
    with open(str(_raw_ckpt), "w", encoding="utf-8") as _f:
        _j.dump(all_samples, _f)
    print(f"\n   [CKPT] raw_samples_v3.json saved ({len(all_samples)} samples)")
except Exception as _e:
    print(f"   [CKPT] save failed: {_e}")


  V3 Strict 1:1 Per-Language Balancing

1. Dedup: 41000 -> 39674
2. Length filter: 39462 samples remain

3. Per-language 1:1 balance:
   Lang              Vuln   Safe  Total  Action
   -------------------------------------------------------
   python             298    298    596  v:ok s:under(5295->298)
   javascript         126    126    252  v:ok s:under(128->126)
   java                97     97    194  v:ok s:ok
   c_cpp            10000  10000  20000  v:under(16136->10000) s:under(17177->10000)
   go                  54     54    108  v:ok s:ok

   Final: 21150 samples — vuln=10575, safe=10575, ratio=1.00:1
   CWE labels known: 21150/21150 (100.0%)

   [CKPT] raw_samples_v3.json saved (21150 samples)


## Cell 5: Exploratory Data Analysis

In [5]:
# ============================================================================
# Cell 5: Exploratory Data Analysis
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

print("="*70)
print(f"  {'Language':<15} {'Total':>7} {'Vuln':>7} {'Safe':>7} {'Ratio':>8}")
print("  " + "-"*55)
lang_stats = {}
for lang in CONFIG["language_ids"]:
    subs = [s for s in all_samples if s["language"]==lang]
    nv = sum(1 for s in subs if s["label"]==1)
    ns = len(subs) - nv
    ratio = nv/max(ns,1)
    lang_stats[lang] = {"total": len(subs), "vuln": nv, "safe": ns, "ratio": ratio}
    print(f"  {lang:<15} {len(subs):>7} {nv:>7} {ns:>7} {ratio:>7.2f}:1")
print("  " + "-"*55)
tot = len(all_samples)
tv_ = sum(1 for s in all_samples if s["label"]==1)
ts_ = tot - tv_
print(f"  {'TOTAL':<15} {tot:>7} {tv_:>7} {ts_:>7} {tv_/max(ts_,1):>7.2f}:1")
print("="*70)

src_ctr = Counter(s.get("source","?") for s in all_samples)
print("\nSource distribution:")
for src, n in src_ctr.most_common():
    print(f"  {src:<25} {n:>5}")

cwe_ctr2 = Counter(s.get("cwe","?") for s in all_samples)
print(f"\nTop 10 CWEs:")
for cwe, n in cwe_ctr2.most_common(10):
    print(f"  {cwe:<20} {n:>5}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
langs_ = [l for l in CONFIG["language_ids"] if lang_stats.get(l,{}).get("total",0)>0]
v_counts = [lang_stats[l]["vuln"] for l in langs_]
s_counts = [lang_stats[l]["safe"] for l in langs_]
x_ = range(len(langs_))
axes[0].bar(x_, v_counts, label="Vulnerable", color="#e74c3c", alpha=.8)
axes[0].bar(x_, s_counts, bottom=v_counts, label="Safe", color="#2ecc71", alpha=.8)
axes[0].set_xticks(list(x_)); axes[0].set_xticklabels(langs_, rotation=20)
axes[0].set_title("Per-Language Balance"); axes[0].legend()
axes[1].bar([c for c,_ in cwe_ctr2.most_common(12)], [n for _,n in cwe_ctr2.most_common(12)], color="#3498db")
axes[1].set_xticklabels([c for c,_ in cwe_ctr2.most_common(12)], rotation=45, ha='right')
axes[1].set_title("Top-12 CWEs")
lens = [len(s["code"].split("\n")) for s in all_samples]
axes[2].hist(lens, bins=40, color="#9b59b6", alpha=.7, edgecolor="black")
axes[2].set_title("Code Length Distribution (lines)"); axes[2].set_xlabel("Lines")
plt.suptitle("V3 Dataset Overview", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "eda_overview_v3.png"), dpi=150)
plt.show()
print("EDA complete.")


  Language          Total    Vuln    Safe    Ratio
  -------------------------------------------------------
  python              596     298     298    1.00:1
  javascript          252     126     126    1.00:1
  java                194      97      97    1.00:1
  c_cpp             20000   10000   10000    1.00:1
  go                  108      54      54    1.00:1
  -------------------------------------------------------
  TOTAL             21150   10575   10575    1.00:1

Source distribution:
  BigVul                     5777
  DiverseVul                 4935
  Juliet-C                   3611
  CrossVul                   3428
  Devign                     3002
  VUDENC                      222
  CVEfixes-Py                 175

Top 10 CWEs:
  unknown               5450
  CWE-119               2178
  CWE-20                1805
  CWE-125               1064
  CWE-264                857
  CWE-200                841
  CWE-399                601
  CWE-189                495
  CWE-416      

/tmp/ipykernel_24/247468078.py:47: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[1].set_xticklabels([c for c,_ in cwe_ctr2.most_common(12)], rotation=45, ha='right')


EDA complete.


## Cell 6: Build Code Graphs (tree-sitter AST + regex CFG/DDG fallback)

In [6]:
# ============================================================================
# Cell 6: Build Code Graphs
# V3: tree-sitter for real AST node extraction, regex for CFG/DDG edges.
# Falls back to pure regex if tree-sitter unavailable.
# ============================================================================
import networkx as nx, re
from collections import Counter

BRANCH_KW = {
    "python":     re.compile(r'^\s*(if|elif|else|for|while|try|except|finally|with|def|class|async)\b'),
    "javascript": re.compile(r'^\s*(if|else|for|while|do|switch|case|try|catch|finally|function|async|class)\b'),
    "java":       re.compile(r'^\s*(if|else|for|while|do|switch|case|try|catch|finally|class|public|private|protected)\b'),
    "c_cpp":      re.compile(r'^\s*(if|else|for|while|do|switch|case|goto|struct|typedef)\b'),
    "go":         re.compile(r'^\s*(if|else|for|switch|case|select|func|go|defer|type)\b'),
}
RETURN_KW = re.compile(r'^\s*(return|raise|throw|panic|break|continue|goto)\b')
VAR_DEF   = re.compile(r'(\b[a-zA-Z_]\w*)\s*(?:=|:=|<-)')
SKIP_VARS = {'if','for','while','return','else','self','this','true','false',
             'nil','null','None','var','let','const','int','string','bool'}

def get_indent(line):
    s = line.lstrip()
    return len(line) - len(s) if s else 0

def _extract_ts_nodes(code: str, language: str):
    # Use tree-sitter to get meaningful AST nodes (statements, expressions).
    if not USE_TREE_SITTER or language not in _TS_LANGS:
        return None
    try:
        parser = Parser(_TS_LANGS[language])
        tree = parser.parse(bytes(code, "utf-8"))
        INTERESTING = {
            "function_definition","function_declaration","method_declaration",
            "if_statement","for_statement","while_statement","return_statement",
            "expression_statement","assignment","call_expression","call",
            "variable_declaration","local_variable_declaration",
            "try_statement","catch_clause","block",
        }
        nodes_text = []
        def walk(node):
            if node.type in INTERESTING:
                text = code[node.start_byte:node.end_byte][:200].replace("\n"," ").strip()
                if text:
                    nodes_text.append(text)
            for child in node.children:
                walk(child)
        walk(tree.root_node)
        return nodes_text[:CONFIG["max_nodes"]] if nodes_text else None
    except Exception:
        return None

def build_code_graph(code: str, language: str) -> nx.DiGraph:
    G = nx.DiGraph()
    # Try tree-sitter first
    ts_nodes = _extract_ts_nodes(code, language)
    if ts_nodes:
        for i, text in enumerate(ts_nodes):
            G.add_node(i, text=text, indent=0)
        # Sequential AST edges + skip connections
        for i in range(len(ts_nodes)-1):
            G.add_edge(i, i+1, type="ast")
        for i in range(len(ts_nodes)-2):
            G.add_edge(i, i+2, type="ast_skip")
        # DDG: simple variable def-use within window
        VAR_DEF2 = re.compile(r'\b([a-zA-Z_]\w*)\s*(?:=|:=)')
        def_sites = {}
        for i, text in enumerate(ts_nodes):
            for var in VAR_DEF2.findall(text):
                if var not in SKIP_VARS:
                    def_sites[var] = i
            for var, def_i in def_sites.items():
                if var in text and def_i != i and abs(i-def_i) <= 5:
                    G.add_edge(def_i, i, type="ddg")
        return G

    # Regex fallback (same as V2)
    lines = code.split('\n')
    non_empty = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped and not stripped.startswith(('#','//','/*','*','*/')):
            non_empty.append((i, line, stripped))
    if not non_empty:
        G.add_node(0, text="empty", indent=0)
        return G
    for idx, (_, line, stripped) in enumerate(non_empty):
        G.add_node(idx, text=stripped, indent=get_indent(line))
    # AST edges
    indent_stack = []
    for idx, (_, line, _) in enumerate(non_empty):
        indent = get_indent(line)
        while indent_stack and indent_stack[-1][1] >= indent:
            indent_stack.pop()
        if indent_stack:
            G.add_edge(indent_stack[-1][0], idx, type="ast")
        indent_stack.append((idx, indent))
    # CFG
    branch_re = BRANCH_KW.get(language, BRANCH_KW["python"])
    for idx in range(len(non_empty)-1):
        stripped = non_empty[idx][2]
        if branch_re.match(stripped) or RETURN_KW.match(stripped):
            G.add_edge(idx, idx+1, type="cfg_branch" if branch_re.match(stripped) else "cfg_return")
        else:
            G.add_edge(idx, idx+1, type="cfg")
    # DDG
    def_sites = {}
    for idx, (_, _, stripped) in enumerate(non_empty):
        for var in VAR_DEF.findall(stripped):
            if var not in SKIP_VARS:
                def_sites[var] = idx
        for var, def_i in def_sites.items():
            if var in stripped and def_i != idx and abs(idx-def_i) <= 8:
                G.add_edge(def_i, idx, type="ddg")
    return G

# Quick test
_test_code = "def foo(x):\n    y = x + 1\n    if y > 0:\n        return y\n    return 0"
_g = build_code_graph(_test_code, "python")
print(f"Graph builder OK: {_g.number_of_nodes()} nodes, {_g.number_of_edges()} edges")
print(f"  Using: {'tree-sitter' if USE_TREE_SITTER else 'regex'} mode")
print(f"  Edge types: {Counter(d.get('type','?') for _,_,d in _g.edges(data=True))}")


Graph builder OK: 8 nodes, 14 edges
  Using: tree-sitter mode
  Edge types: Counter({'ast': 6, 'ast_skip': 5, 'ddg': 3})


## Cell 7: Build PyG Dataset (GraphCodeBERT + structural features, with checkpoint)

In [7]:
# ============================================================================
# Cell 7: Build PyG Dataset with GraphCodeBERT + structural features
# Checkpoint: saves partial progress every 500 graphs.
# ============================================================================
import torch, torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from torch_geometric.data import Data
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading GraphCodeBERT...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["embedding_model"])
gcb_model  = AutoModel.from_pretrained(CONFIG["embedding_model"]).to(device)
gcb_model.eval()
print(f"  Loaded on {device}")

def is_sink(text):   return any(p in text.lower() for p in SINK_PATTERNS)
def is_source(text): return any(p in text.lower() for p in SOURCE_PATTERNS)

def embed_batch(texts, bs=64):
    embs = []
    for i in range(0, len(texts), bs):
        batch = texts[i:i+bs]
        toks = tokenizer(batch, padding=True, truncation=True,
                         max_length=CONFIG["max_tokens_per_node"],
                         return_tensors="pt").to(device)
        with torch.no_grad():
            out = gcb_model(**toks)
            # mean pooling (better than CLS for code; avoids missing pooler issue)
            mask = toks["attention_mask"].unsqueeze(-1).float()
            mean_emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
            embs.append(mean_emb.cpu())
    return torch.cat(embs, dim=0) if embs else torch.empty(0, 768)

def build_pyg(sample, max_nodes=300):
    G = build_code_graph(sample["code"], sample["language"])
    nodes = list(G.nodes(data=True))[:max_nodes]
    if not nodes: return None
    keep = set(n[0] for n in nodes)
    G = G.subgraph(keep).copy()
    mapping = {old: new for new, old in enumerate(sorted(G.nodes()))}
    G = nx.relabel_nodes(G, mapping)
    nodes = list(G.nodes(data=True))
    n = len(nodes)
    texts = [d.get("text","empty") for _, d in nodes]
    gcb_emb = embed_batch(texts, CONFIG["embedding_batch_size"])
    if gcb_emb.shape[0] == 0: return None
    # Structural features
    in_deg  = np.array([G.in_degree(i)  for i in range(n)], dtype=np.float32)
    out_deg = np.array([G.out_degree(i) for i in range(n)], dtype=np.float32)
    in_n  = in_deg  / max(in_deg.max(),  1.)
    out_n = out_deg / max(out_deg.max(), 1.)
    sink_f   = np.array([float(is_sink(texts[i]))   for i in range(n)], dtype=np.float32)
    source_f = np.array([float(is_source(texts[i])) for i in range(n)], dtype=np.float32)
    try:
        lengths = nx.single_source_shortest_path_length(G, 0) if n > 0 else {}
        depth = np.array([lengths.get(i, 0) for i in range(n)], dtype=np.float32)
    except Exception:
        depth = np.zeros(n, dtype=np.float32)
    depth_n = depth / max(depth.max(), 1.)
    lang_id = CONFIG["language_ids"].get(sample["language"], 0.5)
    lang_f  = np.full(n, lang_id, dtype=np.float32)
    struct  = np.stack([in_n, out_n, sink_f, source_f, depth_n, lang_f], axis=1)
    x = torch.cat([gcb_emb, torch.tensor(struct, dtype=torch.float32)], dim=1)
    assert x.shape[1] == CONFIG["input_dim"], f"dim mismatch {x.shape[1]} != {CONFIG['input_dim']}"
    edges = list(G.edges())
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.tensor([[0],[0]], dtype=torch.long)
    data = Data(x=x, edge_index=edge_index, y=torch.tensor(sample["label"], dtype=torch.long))
    data.language   = sample["language"]
    data.cwe        = sample.get("cwe", "unknown")
    data.source_ds  = sample.get("source", "unknown")
    return data

# ── Checkpoint-aware build ─────────────────────────────────────────────────
_pyg_done    = OUTPUT_DIR / "pyg_dataset_v3.pt"
_pyg_partial = OUTPUT_DIR / "pyg_dataset_v3_partial.pt"

if _pyg_done.exists():
    pyg_dataset = torch.load(str(_pyg_done), weights_only=False)
    print(f"[CKPT] Loaded complete v3 dataset: {len(pyg_dataset)} graphs")
else:
    if _pyg_partial.exists():
        pyg_dataset = torch.load(str(_pyg_partial), weights_only=False)
        _start = len(pyg_dataset)
        print(f"[CKPT] Resuming from {_start} graphs")
    else:
        pyg_dataset, _start = [], 0

    print(f"\nBuilding PyG dataset ({len(all_samples)} samples, start={_start})...")
    _SAVE_EVERY = 500
    failed = 0
    for i, sample in enumerate(tqdm(all_samples[_start:], desc="Embedding",
                                    initial=_start, total=len(all_samples))):
        try:
            d = build_pyg(sample, CONFIG["max_nodes"])
            if d is not None: pyg_dataset.append(d)
            else: failed += 1
        except Exception as e:
            failed += 1
            if i < 5: print(f"  Error [{_start+i}]: {e}")
        if len(pyg_dataset) > 0 and len(pyg_dataset) % _SAVE_EVERY == 0:
            try: torch.save(pyg_dataset, str(_pyg_partial))
            except Exception: pass

    print(f"\n  Built {len(pyg_dataset)} graphs, {failed} failed")

    # Save final
    for p in [_pyg_done, OUTPUT_DIR / "pyg_dataset_v3_backup.pt"]:
        try:
            torch.save(pyg_dataset, str(p))
            print(f"  [CKPT] Saved -> {p}")
        except Exception as e:
            print(f"  [CKPT] Save failed ({p}): {e}")
    if _pyg_partial.exists():
        try: _pyg_partial.unlink()
        except Exception: pass

# Free GPU memory
del gcb_model, tokenizer
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("  GraphCodeBERT freed from GPU")

if not pyg_dataset:
    raise RuntimeError(f"[ABORT] 0 graphs built from {len(all_samples)} samples. Check SINK_PATTERNS are defined and datasets loaded correctly.")

if pyg_dataset:
    s0 = pyg_dataset[0]
    print(f"\n  Sample: {s0.x.shape[0]} nodes, {s0.edge_index.shape[1]} edges, dim={s0.x.shape[1]}")
    assert s0.x.shape[1] == CONFIG["input_dim"], f"Input dim mismatch!"
    print(f"  Total graphs: {len(pyg_dataset)}")


Loading GraphCodeBERT...


config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded on cuda

Building PyG dataset (21150 samples, start=0)...



Embedding: 100%|██████████| 21150/21150 [2:19:13<00:00,  2.53it/s]



  Built 21150 graphs, 0 failed
  [CKPT] Saved -> /kaggle/working/pyg_dataset_v3.pt
  [CKPT] Saved -> /kaggle/working/pyg_dataset_v3_backup.pt
  GraphCodeBERT freed from GPU

  Sample: 46 nodes, 89 edges, dim=774
  Total graphs: 21150


## Cell 8: Stratified Split

In [8]:
# ============================================================================
# Cell 8: Stratified Split (by label + language)
# ============================================================================
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

strat_keys = [f"{d.y.item()}_{getattr(d,'language','unk')}" for d in pyg_dataset]
indices = list(range(len(pyg_dataset)))
cal_test_ratio = CONFIG["cal_ratio"] + CONFIG["test_ratio"]

def stratified_split(idx, test_size, keys):
    try:
        return train_test_split(idx, test_size=test_size, stratify=[keys[i] for i in idx], random_state=SEED)
    except ValueError:
        return train_test_split(idx, test_size=test_size, random_state=SEED)

train_val_idx, cal_test_idx = stratified_split(indices, cal_test_ratio, strat_keys)
val_frac  = CONFIG["val_ratio"] / (CONFIG["train_ratio"] + CONFIG["val_ratio"])
test_frac = CONFIG["test_ratio"] / (CONFIG["cal_ratio"]  + CONFIG["test_ratio"])
train_idx, val_idx   = stratified_split(train_val_idx, val_frac,  strat_keys)
cal_idx,   test_idx  = stratified_split(cal_test_idx,  test_frac, strat_keys)

train_data = [pyg_dataset[i] for i in train_idx]
val_data   = [pyg_dataset[i] for i in val_idx]
cal_data   = [pyg_dataset[i] for i in cal_idx]
test_data  = [pyg_dataset[i] for i in test_idx]

print("Split summary:")
for name, split in [("Train", train_data), ("Val", val_data), ("Cal", cal_data), ("Test", test_data)]:
    labels = [d.y.item() for d in split]
    nv_ = sum(labels); ns_ = len(labels) - nv_
    print(f"  {name:<6} {len(split):>5}  vuln={nv_}  safe={ns_}  ratio={nv_/max(ns_,1):.2f}:1")

BS = CONFIG["batch_size"]
train_loader = DataLoader(train_data, batch_size=BS, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_data,   batch_size=BS, shuffle=False)
cal_loader   = DataLoader(cal_data,   batch_size=BS, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BS, shuffle=False)
print(f"\nDataLoaders ready (batch_size={BS})")


Split summary:
  Train  12689  vuln=6345  safe=6344  ratio=1.00:1
  Val     3173  vuln=1586  safe=1587  ratio=1.00:1
  Cal     3172  vuln=1586  safe=1586  ratio=1.00:1
  Test    2116  vuln=1058  safe=1058  ratio=1.00:1

DataLoaders ready (batch_size=64)


## Cell 9: MiniGINv3 — 3-layer GIN with Residual + BatchNorm

In [9]:
# ============================================================================
# Cell 9: MiniGINv3 Model
# Architecture: GIN (Xu et al. 2019) — provably more expressive than GCN/GAT
# 3 layers, hidden=384, residual connections, batch norm, dual pooling.
# V2 used 2-layer GAT (less expressive, 298K params).
# V3 uses 3-layer GIN (1.5M params, residual, batch norm).
# ============================================================================
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GINConv, global_mean_pool, global_add_pool

class MiniGINv3(nn.Module):
    # SEC-C Mini-GIN V3.
    # Input: 774-dim (768 GCB mean-pool + 6 structural)
    # Arch:  Lin(774->H) -> GINx3(H->H, residual, BN) -> MeanPool+AddPool -> Clf(2H->out)
    def __init__(self, input_dim=774, hidden_dim=384, dropout=0.35, num_classes=2):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(input_dim, H)
        self.bn_in      = nn.BatchNorm1d(H)

        # 3 GIN layers with MLP (2-layer MLP per GIN)
        self.gins = nn.ModuleList()
        self.bns  = nn.ModuleList()
        for _ in range(3):
            mlp = nn.Sequential(
                nn.Linear(H, H * 2),
                nn.BatchNorm1d(H * 2),
                nn.ReLU(),
                nn.Dropout(dropout / 2),
                nn.Linear(H * 2, H),
            )
            self.gins.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(H))

        self.dropout = nn.Dropout(dropout)

        # Dual pooling: mean + add → 2H-dim graph embedding
        pool_dim = H * 2

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(pool_dim, H),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes),
        )
        # Confidence estimation head
        self.confidence_head = nn.Sequential(
            nn.Linear(pool_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x, edge_index, batch):
        h = F.relu(self.bn_in(self.input_proj(x)))

        # 3 GIN layers with residual
        prev = h
        for gin, bn in zip(self.gins, self.bns):
            h_new = bn(gin(h, edge_index))
            h_new = F.relu(h_new)
            h_new = self.dropout(h_new)
            h = h_new + prev        # residual
            prev = h

        # Dual pooling (mean + add)
        mean_pool = global_mean_pool(h, batch)
        add_pool  = global_add_pool(h, batch)
        graph_emb = torch.cat([mean_pool, add_pool], dim=-1)

        logits     = self.classifier(graph_emb)
        confidence = self.confidence_head(graph_emb)
        return logits, confidence


model = MiniGINv3(
    input_dim=CONFIG["input_dim"],
    hidden_dim=CONFIG["hidden_dim"],
    dropout=CONFIG["dropout"],
    num_classes=CONFIG["num_classes"],
).to(device)

pc = {"total": sum(p.numel() for p in model.parameters()),
      "trainable": sum(p.numel() for p in model.parameters() if p.requires_grad)}
print(f"MiniGINv3 on {device}")
print(f"  Params: {pc['trainable']:,} (V2 was 298K)")
print(f"  Arch:   {CONFIG['input_dim']} -> {CONFIG['hidden_dim']}*3 -> {CONFIG['hidden_dim']*2} -> 2")


MiniGINv3 on cuda
  Params: 2,375,046 (V2 was 298K)
  Arch:   774 -> 384*3 -> 768 -> 2


## Cell 10: Training — WeightedCE + LR Warmup + Early Stopping on val F1

Design choices (all corrected from V2, V4 adjustments noted):

| Component | Choice | Reason |
|---|---|---|
| Loss | `CrossEntropyLoss(weight=[1.0, 1.5], label_smoothing=0.1)` | No Focal Loss (caused collapse in V2) |
| Optimizer | AdamW, LR=3e-4, weight_decay=1e-3 | Stable with GCB embeddings |
| LR schedule | Linear warmup (5 ep) → cosine decay | Prevents cold-start instability |
| Early stopping | **val F1**, patience=20 *(V4: was 25)* | V2 used val_loss and stopped 15 ep early |
| Gradient clip | max_norm=0.5 | Prevents exploding gradients |
| Confidence head | BCE auxiliary loss (0.1 weight) | Calibrates sigmoid output for conformal |

With ~13,800 training graphs, expect best val F1 around epoch 25–40.
Training time: ~35–50 min on T4 GPU.


In [10]:
# ============================================================================
# Cell 10: Training
# V3 key changes:
#   - Standard CrossEntropy with mild class weight (NOT focal loss)
#   - LR=3e-4 with 5-epoch linear warmup + cosine decay (NOT raw cosine from ep1)
#   - batch_size=64 (smoother gradients)
#   - patience=25 (give model time to learn)
#   - label_smoothing=0.1 (prevents overconfidence)
# ============================================================================
import copy, time
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

# ── Loss: WeightedCE + label smoothing ────────────────────────────────────
# Mild vuln weight (1.5) instead of inverse frequency (1.74).
# Focal loss caused threshold collapse in V2 — do not use.
class_weights = torch.tensor([1.0, CONFIG["class_weight_vuln"]], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG["label_smoothing"],
)
print(f"Loss: CrossEntropyLoss(weight=[1.0, {CONFIG['class_weight_vuln']}], smoothing={CONFIG['label_smoothing']})")

# ── Optimizer: AdamW ───────────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

# ── LR schedule: linear warmup → cosine decay ─────────────────────────────
def get_lr(epoch):
    warmup = CONFIG["lr_warmup_epochs"]
    total  = CONFIG["epochs"]
    if epoch <= warmup:
        return epoch / max(warmup, 1)     # linear ramp
    progress = (epoch - warmup) / max(total - warmup, 1)
    return 0.5 * (1 + np.cos(np.pi * progress))  # cosine

import math as _math
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr)

# ── Metrics helper ────────────────────────────────────────────────────────
def compute_metrics(preds, labels):
    if not preds: return {"accuracy":0,"precision":0,"recall":0,"f1":0}
    tp = sum(1 for p,l in zip(preds,labels) if p==1 and l==1)
    fp = sum(1 for p,l in zip(preds,labels) if p==1 and l==0)
    fn = sum(1 for p,l in zip(preds,labels) if p==0 and l==1)
    acc  = sum(1 for p,l in zip(preds,labels) if p==l) / len(preds)
    prec = tp / (tp+fp) if (tp+fp) > 0 else 0.
    rec  = tp / (tp+fn) if (tp+fn) > 0 else 0.
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.
    return {"accuracy":acc,"precision":prec,"recall":rec,"f1":f1}

def train_one_epoch():
    model.train()
    total_loss, nb = 0., 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        logits, confidence = model(data.x, data.edge_index, data.batch)
        loss = criterion(logits, data.y)
        with torch.no_grad():
            correct = (logits.argmax(-1) == data.y).float()
        loss += 0.1 * F.binary_cross_entropy(confidence.squeeze(-1), correct)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()
        total_loss += loss.item(); nb += 1
    return total_loss / max(nb, 1)

@torch.no_grad()
def validate(loader):
    model.eval()
    total_loss, nb = 0., 0
    all_preds, all_labels = [], []
    for data in loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        total_loss += criterion(logits, data.y).item(); nb += 1
        all_preds.extend(logits.argmax(-1).cpu().tolist())
        all_labels.extend(data.y.cpu().tolist())
    return total_loss / max(nb, 1), compute_metrics(all_preds, all_labels)

# ── Training loop ─────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  Training MiniGINv3 ({CONFIG['epochs']} epochs, patience={CONFIG['patience']})")
print(f"  LR={CONFIG['lr']}, warmup={CONFIG['lr_warmup_epochs']}ep, batch={CONFIG['batch_size']}")
print(f"{'='*70}\n")

history = {k: [] for k in ["train_loss","val_loss","val_accuracy","val_f1","val_precision","val_recall"]}
best_val_f1, best_val_loss, best_epoch = 0., float('inf'), 0
patience_ctr = 0
best_state   = None
t0 = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss = train_one_epoch()
    scheduler.step()
    val_loss, vm = validate(val_loader)
    for k in ["accuracy","f1","precision","recall"]:
        history[f"val_{k}"].append(vm[k])
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    improved = ""
    if vm["f1"] > best_val_f1:
        best_val_f1 = vm["f1"]
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_ctr = 0
        improved = " *"
        # Save best checkpoint
        try:
            torch.save(best_state, str(OUTPUT_DIR / "mini_gat_v3_best.pt"))
        except Exception: pass
    else:
        patience_ctr += 1

    # Periodic checkpoint every 10 epochs
    if epoch % 10 == 0:
        try:
            torch.save(model.state_dict(), str(OUTPUT_DIR / f"mini_gat_v3_epoch{epoch:03d}.pt"))
            # Prune old periodic checkpoints
            old = OUTPUT_DIR / f"mini_gat_v3_epoch{epoch-20:03d}.pt"
            if old.exists() and not improved: old.unlink()
        except Exception: pass

    lr = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or epoch <= 5 or improved:
        print(f"  Epoch {epoch:3d}/{CONFIG['epochs']}  "
              f"tl={train_loss:.4f} vl={val_loss:.4f} "
              f"F1={vm['f1']:.4f} P={vm['precision']:.4f} R={vm['recall']:.4f} "
              f"lr={lr:.2e}{improved}")

    if patience_ctr >= CONFIG["patience"]:
        print(f"\n  Early stopping at epoch {epoch} (patience={CONFIG['patience']})")
        break

elapsed = time.time() - t0
print(f"\n  Done in {elapsed/60:.1f} min. Best epoch {best_epoch} (F1={best_val_f1:.4f})")
if best_state:
    model.load_state_dict(best_state)
    print("  Restored best weights")

# Plot training curves
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = range(1, len(history["train_loss"]) + 1)
axes[0].plot(epochs_x, history["train_loss"], label="Train"); axes[0].plot(epochs_x, history["val_loss"], label="Val")
axes[0].axvline(best_epoch, color='g', ls='--', alpha=.5, label=f'Best(ep{best_epoch})')
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=.3)
axes[1].plot(epochs_x, history["val_f1"], label="F1", color='purple')
axes[1].plot(epochs_x, history["val_precision"], label="Precision", alpha=.7)
axes[1].plot(epochs_x, history["val_recall"], label="Recall", alpha=.7)
axes[1].set_title("Validation Metrics"); axes[1].legend(); axes[1].grid(True, alpha=.3)
axes[2].plot(epochs_x, history["val_accuracy"], label="Accuracy", color='teal')
axes[2].set_title("Validation Accuracy"); axes[2].legend(); axes[2].grid(True, alpha=.3)
plt.suptitle("MiniGINv3 Training Curves", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "training_curves_v3.png"), dpi=150)
plt.show()


Loss: CrossEntropyLoss(weight=[1.0, 1.5], smoothing=0.0)

  Training MiniGINv3 (100 epochs, patience=20)
  LR=0.0003, warmup=5ep, batch=64

  Epoch   1/100  tl=36.7977 vl=27.9773 F1=0.0000 P=0.0000 R=0.0000 lr=6.00e-05
  Epoch   2/100  tl=5.7623 vl=1.4606 F1=0.7140 P=0.6091 R=0.8625 lr=1.20e-04 *
  Epoch   3/100  tl=1.6252 vl=0.5855 F1=0.7336 P=0.6253 R=0.8871 lr=1.80e-04 *
  Epoch   4/100  tl=1.4025 vl=0.5504 F1=0.7369 P=0.6091 R=0.9325 lr=2.40e-04 *
  Epoch   5/100  tl=1.2181 vl=0.5452 F1=0.7381 P=0.6083 R=0.9382 lr=3.00e-04 *
  Epoch   6/100  tl=1.0683 vl=0.6704 F1=0.7417 P=0.6353 R=0.8909 lr=3.00e-04 *
  Epoch   8/100  tl=0.7157 vl=0.5352 F1=0.7479 P=0.6380 R=0.9035 lr=2.99e-04 *
  Epoch  10/100  tl=0.6297 vl=0.5594 F1=0.7392 P=0.6391 R=0.8764 lr=2.98e-04
  Epoch  12/100  tl=0.5922 vl=0.6000 F1=0.7596 P=0.6404 R=0.9332 lr=2.96e-04 *
  Epoch  15/100  tl=0.5244 vl=0.6058 F1=0.7575 P=0.6323 R=0.9445 lr=2.92e-04
  Epoch  20/100  tl=0.4440 vl=0.5774 F1=0.7585 P=0.6503 R=0.9098 lr=2.82e-

## Cell 11: F1-Optimal Decision Threshold Search

Grid search over 81 thresholds in [0.05, 0.95] on the **validation set** to find
the probability cutoff that maximises F1 score.

Why not argmax (threshold=0.50)?
- V3: threshold=0.41 gave val F1=0.679 vs 0.665 at 0.50 — +1.4 pp
- With imbalanced training data, the model's calibration shifts; optimal threshold
  compensates without retraining

V4 expectation: with better class balance and more data, threshold should be
closer to 0.45–0.50 (more calibrated model).

Saved to: `decision_threshold_v3.json` (loaded by `src/graph/gnn/graph_validator.py`)


In [11]:
# ============================================================================
# Cell 11: F1-Optimal Decision Threshold Calibration (V3 key fix)
#
# V2 bug: always used argmax (equivalent to threshold=0.5), which caused
# high recall / low precision ("predict everything as vulnerable").
#
# V3 fix: search validation set for the threshold T* that maximises F1.
# All downstream predictions (test evaluation, conformal) use T*.
# ============================================================================
import numpy as np
import torch.nn.functional as F

model.eval()
val_probs_all, val_true_all = [], []
with torch.no_grad():
    for data in val_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()   # P(vulnerable)
        val_probs_all.extend(probs.tolist())
        val_true_all.extend(data.y.cpu().tolist())

val_probs_arr = np.array(val_probs_all)
val_true_arr  = np.array(val_true_all)

thresholds = np.linspace(0.05, 0.95, CONFIG["threshold_search_steps"])
results = []
for thr in thresholds:
    preds = (val_probs_arr >= thr).astype(int)
    tp = int(((preds==1) & (val_true_arr==1)).sum())
    fp = int(((preds==1) & (val_true_arr==0)).sum())
    fn = int(((preds==0) & (val_true_arr==1)).sum())
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    results.append({"thr": float(thr), "f1": f1, "prec": prec, "rec": rec, "tp": tp, "fp": fp, "fn": fn})

best_thr_row = max(results, key=lambda r: r["f1"])
DECISION_THRESHOLD = best_thr_row["thr"]

print(f"{'='*60}")
print(f"  F1-Optimal Threshold Search (val set, {len(val_true_arr)} samples)")
print(f"{'='*60}")
print(f"  Best threshold: {DECISION_THRESHOLD:.2f}")
print(f"  Val F1:         {best_thr_row['f1']:.4f}")
print(f"  Val Precision:  {best_thr_row['prec']:.4f}")
print(f"  Val Recall:     {best_thr_row['rec']:.4f}")
print(f"\n  Comparison (V2 used argmax = 0.5):")
row_05 = next(r for r in results if abs(r["thr"] - 0.50) < 0.01)
print(f"  threshold=0.50: F1={row_05['f1']:.4f} P={row_05['prec']:.4f} R={row_05['rec']:.4f}")
print(f"  threshold={DECISION_THRESHOLD:.2f}: F1={best_thr_row['f1']:.4f} P={best_thr_row['prec']:.4f} R={best_thr_row['rec']:.4f}")

# Plot threshold curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
thrs  = [r["thr"]  for r in results]
f1s   = [r["f1"]   for r in results]
precs = [r["prec"] for r in results]
recs  = [r["rec"]  for r in results]
axes[0].plot(thrs, f1s,   label="F1",        color="purple", lw=2)
axes[0].plot(thrs, precs, label="Precision",  color="blue",   alpha=.7)
axes[0].plot(thrs, recs,  label="Recall",     color="red",    alpha=.7)
axes[0].axvline(DECISION_THRESHOLD, color="green", ls="--", lw=2, label=f"Best T={DECISION_THRESHOLD:.2f}")
axes[0].axvline(0.50, color="gray", ls=":", label="V2 (0.5)")
axes[0].set_title("Threshold vs Metrics (Val Set)"); axes[0].legend(); axes[0].grid(True, alpha=.3)
axes[1].hist(val_probs_arr[val_true_arr==0], bins=30, alpha=.6, color="green", label="safe")
axes[1].hist(val_probs_arr[val_true_arr==1], bins=30, alpha=.6, color="red",   label="vulnerable")
axes[1].axvline(DECISION_THRESHOLD, color="black", ls="--", lw=2, label=f"T={DECISION_THRESHOLD:.2f}")
axes[1].set_title("Predicted P(vuln) Distribution"); axes[1].legend()
plt.suptitle("Decision Threshold Calibration", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "threshold_calibration_v3.png"), dpi=150)
plt.show()


  F1-Optimal Threshold Search (val set, 3173 samples)
  Best threshold: 0.32
  Val F1:         0.7644
  Val Precision:  0.6545
  Val Recall:     0.9187

  Comparison (V2 used argmax = 0.5):
  threshold=0.50: F1=0.7623 P=0.6668 R=0.8897
  threshold=0.32: F1=0.7644 P=0.6545 R=0.9187


## Cell 12: Test Set Evaluation

Evaluates on the held-out test set (10% of data, stratified, never seen during
training or conformal calibration).

Reports:
- **Overall:** Accuracy, Precision, Recall, F1, AUC-ROC
- **Per-language:** C/C++, Python, Java breakdowns
- **Per-CWE:** F1 for top-15 CWEs by sample count
- **Plots:** ROC curve, confusion matrix → `evaluation_plots_v3.png`

V4 targets: **F1 >= 0.68**, AUC-ROC >= 0.70, Precision >= 0.63.

C/C++ is the primary evaluation language (~95% of test samples).
Per-CWE results for known CWEs (CWE-119, CWE-20, CWE-125, etc.) should
exceed F1=0.80 based on V3 results (CWE-119: 0.941, CWE-20: 0.909).


In [12]:
# ============================================================================
# Cell 12: Comprehensive Evaluation using DECISION_THRESHOLD (not argmax)
# ============================================================================
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from collections import Counter

CLASS_LABELS = ["safe", "vulnerable"]
model.eval()
test_preds, test_labels = [], []
test_probs, test_langs, test_cwes = [], [], []

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        sm_probs = F.softmax(logits, dim=-1).cpu().numpy()
        vuln_probs = sm_probs[:, 1]
        # Use F1-optimal threshold (not argmax/0.5)
        preds = (vuln_probs >= DECISION_THRESHOLD).astype(int).tolist()
        test_preds.extend(preds)
        test_labels.extend(data.y.cpu().tolist())
        test_probs.append(sm_probs)
        # Per-graph language extraction (from last feature dim)
        batch_np = data.batch.cpu().numpy()
        for b in range(logits.shape[0]):
            mask = batch_np == b
            nidx = np.where(mask)[0]
            if len(nidx) > 0:
                lf = data.x[nidx[0], -1].item()
                ln = min(CONFIG["language_ids"], key=lambda k: abs(CONFIG["language_ids"][k]-lf))
            else:
                ln = "unknown"
            test_langs.append(ln)
        # CWE: PyG batches string attrs as list (one per graph)
        _batch_cwes = data.cwe if hasattr(data, "cwe") else None
        if _batch_cwes is None:
            test_cwes.extend(["unknown"] * logits.shape[0])
        elif isinstance(_batch_cwes, list):
            test_cwes.extend([str(c) if c else "unknown" for c in _batch_cwes])
        else:
            test_cwes.extend([str(_batch_cwes)] * logits.shape[0])

test_probs_np = np.concatenate(test_probs, axis=0)
test_metrics  = compute_metrics(test_preds, test_labels)

print("="*70)
print(f"  TEST SET EVALUATION (threshold={DECISION_THRESHOLD:.2f}, N={len(test_labels)})")
print("="*70)
print(f"  Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"  Precision: {test_metrics['precision']:.4f}")
print(f"  Recall:    {test_metrics['recall']:.4f}")
print(f"  F1:        {test_metrics['f1']:.4f}")
try:
    auc_roc = roc_auc_score(test_labels, test_probs_np[:, 1])
    print(f"  AUC-ROC:   {auc_roc:.4f}")
except Exception:
    auc_roc = 0.
    print("  AUC-ROC:   N/A")

print(f"\n{classification_report(test_labels, test_preds, target_names=CLASS_LABELS)}")

# Per-language
print(f"\n  Per-Language:")
print(f"  {'Lang':<15} {'N':>5} {'Acc':>7} {'P':>7} {'R':>7} {'F1':>7}")
print(f"  {'-'*50}")
lang_metrics = {}
for lang in CONFIG["language_ids"]:
    mi = [i for i,l in enumerate(test_langs) if l==lang]
    if not mi: continue
    m = compute_metrics([test_preds[i] for i in mi], [test_labels[i] for i in mi])
    lang_metrics[lang] = m
    print(f"  {lang:<15} {len(mi):>5} {m['accuracy']:>7.4f} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f1']:>7.4f}")

# Per-CWE
print(f"\n  Per-CWE (top 15):")
cwe_ctr = Counter(str(c) for c in test_cwes)
print(f"  {'CWE':<22} {'N':>5} {'F1':>7}")
print(f"  {'-'*37}")
for cwe, _ in cwe_ctr.most_common(15):
    ci = [i for i,c in enumerate(test_cwes) if str(c)==cwe]
    if len(ci) < 3: continue
    m = compute_metrics([test_preds[i] for i in ci], [test_labels[i] for i in ci])
    print(f"  {cwe:<22} {len(ci):>5} {m['f1']:>7.4f}")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm = confusion_matrix(test_labels, test_preds)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_title(f"Confusion Matrix (T={DECISION_THRESHOLD:.2f})")
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(CLASS_LABELS); axes[0].set_yticklabels(CLASS_LABELS)
for ii in range(2):
    for jj in range(2):
        axes[0].text(jj,ii,str(cm[ii,jj]),ha='center',va='center',
                     color='white' if cm[ii,jj]>cm.max()/2 else 'black',fontsize=14)
plt.colorbar(im, ax=axes[0])
try:
    fpr, tpr, _ = roc_curve(test_labels, test_probs_np[:,1])
    axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc_roc:.4f}')
    axes[1].plot([0,1],[0,1],'r--',alpha=.5); axes[1].set_title("ROC"); axes[1].legend()
except Exception:
    axes[1].text(.5,.5,"N/A",ha='center')
axes[2].hist(test_probs_np[np.array(test_labels)==0,1], bins=30, alpha=.6, color='green', label='safe')
axes[2].hist(test_probs_np[np.array(test_labels)==1,1], bins=30, alpha=.6, color='red',   label='vulnerable')
axes[2].axvline(DECISION_THRESHOLD, color='black', ls='--', lw=2, label=f'T={DECISION_THRESHOLD:.2f}')
axes[2].set_title("Score Distribution"); axes[2].legend()
plt.suptitle("MiniGINv3 Test Evaluation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "evaluation_plots_v3.png"), dpi=150)
plt.show()


  TEST SET EVALUATION (threshold=0.32, N=2116)
  Accuracy:  0.6999
  Precision: 0.6430
  Recall:    0.8989
  F1:        0.7497
  AUC-ROC:   0.7813

              precision    recall  f1-score   support

        safe       0.83      0.50      0.63      1058
  vulnerable       0.64      0.90      0.75      1058

    accuracy                           0.70      2116
   macro avg       0.74      0.70      0.69      2116
weighted avg       0.74      0.70      0.69      2116


  Per-Language:
  Lang                N     Acc       P       R      F1
  --------------------------------------------------
  python             60  0.8167  0.7568  0.9333  0.8358
  javascript         26  0.5385  0.5200  1.0000  0.6842
  java               20  0.5000  0.5000  1.0000  0.6667
  c_cpp            2000  0.7020  0.6455  0.8960  0.7504
  go                 10  0.4000  0.4444  0.8000  0.5714

  Per-CWE (top 15):
  CWE                        N      F1
  -------------------------------------
  unknown          

## Cell 12b: Conformal Temperature Scaling (ConfTS)

Post-hoc temperature optimization to minimize APS prediction set sizes (Dabah et al. 2024, arXiv:2402.04344).

- Uses **val set** for T optimization, **cal set** for final conformal threshold
- Preserves exchangeability → coverage guarantee maintained (Kofman et al. ICML 2025)
- T < 1 sharpens softmax outputs, enabling singleton prediction sets

In [13]:
# ============================================================================
# Cell 12b: Conformal Temperature Scaling (ConfTS)
# V5 NEW: Post-hoc temperature optimization to minimize APS set sizes
# while maintaining coverage >= 1-alpha.
#
# Theory: ConfTS (Dabah et al. 2024, arXiv:2402.04344) optimizes a single
# scalar T such that softmax(logits / T) produces smaller conformal
# prediction sets. T < 1 sharpens confident predictions; T > 1 softens.
# Using val set for T search and cal set for final threshold preserves
# exchangeability -> coverage guarantee maintained.
# ============================================================================
import math
import numpy as np
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# -- APS helper functions (used here and in conformal cell) ------------------
def aps_scores(sm_probs, true_labels):
    """Compute APS nonconformity scores."""
    n = len(true_labels)
    scores = np.zeros(n, np.float64)
    for i in range(n):
        probs = sm_probs[i]
        si = np.argsort(-probs)
        cs = np.cumsum(probs[si])
        rank = int(np.where(si == true_labels[i])[0][0])
        scores[i] = cs[rank]
    return scores

CLASS_LABELS = ["safe", "vulnerable"]

def build_pred_set(probs, threshold):
    """Build a conformal prediction set from softmax probs and threshold."""
    si = np.argsort(-probs)
    cs = np.cumsum(probs[si])
    ps = []
    for j, idx in enumerate(si):
        ps.append(CLASS_LABELS[int(idx)])
        if cs[j] >= threshold: break
    return ps or [CLASS_LABELS[int(si[0])]]

# -- Collect raw logits from val set ----------------------------------------
alpha = CONFIG["alpha"]
model.eval()

val_logits_list, val_labels_list = [], []
with torch.no_grad():
    for data in val_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        val_logits_list.append(logits.cpu().numpy())
        val_labels_list.extend(data.y.cpu().tolist())

val_logits_np = np.concatenate(val_logits_list, axis=0)
val_labels_np = np.array(val_labels_list, dtype=np.int64)
n_val = len(val_labels_np)

# -- Grid search over temperature T -----------------------------------------
T_candidates = np.concatenate([
    np.arange(0.05, 0.5, 0.05),   # aggressive sharpening
    np.arange(0.5, 1.5, 0.02),    # fine-grained around T=1
    np.arange(1.5, 3.01, 0.1),    # softening
])

print(f"{'='*60}")
print(f"  ConfTS: Temperature Search ({len(T_candidates)} candidates)")
print(f"  alpha={alpha}, val_n={n_val}")
print(f"{'='*60}")

best_T, best_mean_size = 1.0, 2.0
results_T = []

for T in T_candidates:
    # Tempered softmax on val set
    scaled = val_logits_np / T
    exp_s = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    val_sm = exp_s / exp_s.sum(axis=1, keepdims=True)

    # APS scores on val set
    scores_v = aps_scores(val_sm, val_labels_np)

    # Quantile threshold
    ql = min(math.ceil((n_val + 1) * (1. - alpha)) / n_val, 1.)
    try:
        thr_v = float(np.quantile(scores_v, ql, method="higher"))
    except TypeError:
        thr_v = float(np.quantile(scores_v, ql, interpolation="higher"))

    # Coverage and set sizes on val set
    covered, singleton, total_size = 0, 0, 0
    for i in range(n_val):
        ps = build_pred_set(val_sm[i], thr_v)
        total_size += len(ps)
        if CLASS_LABELS[val_labels_np[i]] in ps:
            covered += 1
        if len(ps) == 1:
            singleton += 1

    cov = covered / n_val
    mean_sz = total_size / n_val
    sing_rate = singleton / n_val

    results_T.append({
        "T": float(T), "coverage": cov, "mean_size": mean_sz,
        "singleton_rate": sing_rate, "threshold": thr_v
    })

    # Select: must maintain coverage, minimize mean set size
    if cov >= (1 - alpha) and mean_sz < best_mean_size:
        best_T = float(T)
        best_mean_size = mean_sz

CONFORMAL_TEMPERATURE = best_T
best_row = next(r for r in results_T if abs(r["T"] - best_T) < 0.001)
baseline_row = min(results_T, key=lambda r: abs(r["T"] - 1.0))

print(f"\n  Best T = {CONFORMAL_TEMPERATURE:.3f}")
print(f"  Mean set size: {best_row['mean_size']:.4f}  (T=1.0 baseline: {baseline_row['mean_size']:.4f})")
print(f"  Singleton rate: {best_row['singleton_rate']:.1%}  (T=1.0: {baseline_row['singleton_rate']:.1%})")
print(f"  Coverage: {best_row['coverage']:.4f}  (target >= {1-alpha:.2f})")

# Show top candidates
print(f"\n  {'T':>6} {'Cov':>7} {'MeanSz':>8} {'Single%':>8} {'Thr':>8}")
print(f"  {'-'*42}")
for r in sorted(results_T, key=lambda x: x["mean_size"])[:12]:
    flag = " <--" if abs(r["T"] - CONFORMAL_TEMPERATURE) < 0.001 else ""
    print(f"  {r['T']:6.3f} {r['coverage']:7.4f} {r['mean_size']:8.4f} "
          f"{r['singleton_rate']:7.1%} {r['threshold']:8.4f}{flag}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
Ts = [r["T"] for r in results_T]
axes[0].plot(Ts, [r["mean_size"] for r in results_T], "b-", lw=2, label="Mean set size")
axes[0].axvline(CONFORMAL_TEMPERATURE, color="green", ls="--", lw=2,
                label=f"Best T={CONFORMAL_TEMPERATURE:.3f}")
axes[0].axvline(1.0, color="gray", ls=":", label="T=1.0 (no scaling)")
axes[0].set_xlabel("Temperature T"); axes[0].set_ylabel("Mean Set Size")
axes[0].set_title("ConfTS: Temperature vs Set Size"); axes[0].legend(); axes[0].grid(True, alpha=.3)

axes[1].plot(Ts, [r["singleton_rate"] for r in results_T], "purple", lw=2, label="Singleton rate")
axes[1].plot(Ts, [r["coverage"] for r in results_T], "r--", lw=1.5, label="Coverage")
axes[1].axhline(1-alpha, color="orange", ls=":", label=f"1-alpha={1-alpha:.2f}")
axes[1].axvline(CONFORMAL_TEMPERATURE, color="green", ls="--", lw=2)
axes[1].set_xlabel("Temperature T"); axes[1].set_title("ConfTS: Singleton Rate & Coverage")
axes[1].legend(); axes[1].grid(True, alpha=.3)
plt.suptitle("Conformal Temperature Scaling (ConfTS, Dabah et al. 2024)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "confts_temperature_v5.png"), dpi=150)
plt.show()
print(f"\n  Saved confts_temperature_v5.png")

  ConfTS: Temperature Search (75 candidates)
  alpha=0.1, val_n=3173

  Best T = 0.100
  Mean set size: 1.4630  (T=1.0 baseline: 1.9738)
  Singleton rate: 53.7%  (T=1.0: 2.6%)
  Coverage: 0.9203  (target >= 0.90)

       T     Cov   MeanSz  Single%      Thr
  ------------------------------------------
   0.050  0.8711   1.3382   66.2%   1.0000
   0.100  0.9203   1.4630   53.7%   1.0000 <--
   0.150  0.9436   1.5424   45.8%   1.0000
   0.200  0.9628   1.6089   39.1%   1.0000
   0.250  0.9764   1.6618   33.8%   1.0000
   0.300  0.9830   1.7094   29.1%   1.0000
   0.350  0.9861   1.7469   25.3%   1.0000
   0.400  0.9899   1.7816   21.8%   1.0000
   0.450  0.9915   1.8424   15.8%   1.0000
   0.500  0.9928   1.8698   13.0%   1.0000
   0.520  0.9934   1.8783   12.2%   1.0000
   0.540  0.9934   1.8834   11.7%   1.0000

  Saved confts_temperature_v5.png


## Cell 13: Conformal Prediction — APS (alpha=0.1, 90% coverage guarantee)

**Adaptive Prediction Sets** (Angelopoulos & Bates, ICLR 2024).
First application to vulnerability detection — novel contribution of SEC-C.

**V4 fix:** `alpha=0.1` (was 0.2 in V3, which gave only 80% coverage).
Now aligned with framework `configs/default.yaml` (`conformal.alpha: 0.1`).

**Coverage guarantee:** P(true label in prediction_set) >= 1 - alpha = **0.90**,
distribution-free, holds for any data distribution without parametric assumptions.

### Algorithm

**Calibration phase** (on ~454 held-out cal samples):
1. Run model forward pass → softmax probabilities
2. Sort classes by descending probability
3. Compute cumulative sum; record value at position of true label = nonconformity score
4. Set threshold: `q_hat = quantile(scores, ceil((n+1)(1-alpha)) / n)`

**Inference phase** (per finding at runtime):
1. Sort classes by descending softmax probability
2. Include classes until cumulative sum >= q_hat
3. Return resulting set:
   - `{"safe"}` → confident SAFE, resolve at Stage 2 (no LLM)
   - `{"vulnerable"}` → confident VULNERABLE, resolve at Stage 2 as LIKELY
   - `{"safe", "vulnerable"}` → ambiguous, escalate to Stage 3 (LLM dual-agent)

### V3 vs V4 Conformal Expectations

| Metric | V3 (actual) | V4 (target) |
|---|---|---|
| Threshold | 1.0 (broken) | **0.78 – 0.92** |
| Singleton rate | 0.2% | **20 – 40%** |
| Ambiguous rate | 99.7% | **60 – 80%** |
| Empirical coverage | 1.0 (trivial) | **>= 0.90** (meaningful) |

V3 failure cause: 2.37M-param model overfit on 1,819 samples → uncertain on cal set
→ all scores pile at 1.0 → threshold must be 1.0. V4 fixes this via 8x more data.

Saved to: `conformal_calibration_v3.json`


In [14]:
# ============================================================================
# Cell 13: Conformal Prediction — APS calibration
# V5: alpha=0.1, uses ConfTS temperature scaling for smaller prediction sets.
# NOTE: conformal uses softmax probs (not threshold), so even if classification
# threshold is calibrated, conformal adds a set-valued guarantee.
# ============================================================================
import math

# aps_scores, build_pred_set, CLASS_LABELS defined in ConfTS cell above

# Collect cal probabilities
alpha = CONFIG["alpha"]
print(f"{'='*60}")
print(f"  Conformal Prediction (APS, alpha={alpha})")
print(f"{'='*60}")

model.eval()
cal_sm, cal_lb = [], []
with torch.no_grad():
    for data in cal_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        # V5: Apply conformal temperature scaling
        cal_sm.append(F.softmax(logits / CONFORMAL_TEMPERATURE, dim=-1).cpu().numpy())
        cal_lb.extend(data.y.cpu().tolist())

cal_sm   = np.concatenate(cal_sm, axis=0)
cal_lb_np = np.array(cal_lb, dtype=np.int64)
n_cal    = len(cal_lb_np)

scores = aps_scores(cal_sm, cal_lb_np)
ql = min(math.ceil((n_cal + 1) * (1. - alpha)) / n_cal, 1.)
try:
    conf_threshold = float(np.quantile(scores, ql, method="higher"))
except TypeError:
    conf_threshold = float(np.quantile(scores, ql, interpolation="higher"))

print(f"\n  Cal samples:  {n_cal}")
print(f"  Temperature:  {CONFORMAL_TEMPERATURE:.4f}  (ConfTS optimized)")
print(f"  Threshold:    {conf_threshold:.4f}  (V2 was 1.0 — broken)")
print(f"  Score mean:   {scores.mean():.4f}")
print(f"  Score std:    {scores.std():.4f}")

# Calibration coverage
covered, single, sizes = 0, 0, []
for i in range(n_cal):
    ps = build_pred_set(cal_sm[i], conf_threshold)
    sizes.append(len(ps))
    if CLASS_LABELS[cal_lb_np[i]] in ps: covered += 1
    if len(ps) == 1: single += 1

empirical_coverage = covered / n_cal
singleton_rate     = single / n_cal
ambig_rate         = 1 - singleton_rate
print(f"\n  [Calibration]")
print(f"  Coverage:     {empirical_coverage:.4f}  (target>={1-alpha:.2f})")
print(f"  Singleton:    {single}/{n_cal} ({singleton_rate:.1%})")
print(f"  Ambiguous:    {n_cal-single}/{n_cal} ({ambig_rate:.1%})")

# Test verification (V5: use T-scaled softmax for test set too)
test_sm_t = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        logits, _ = model(data.x, data.edge_index, data.batch)
        test_sm_t.append(F.softmax(logits / CONFORMAL_TEMPERATURE, dim=-1).cpu().numpy())
test_sm_conformal = np.concatenate(test_sm_t, axis=0)

tc, ts, tsz = 0, 0, []
for i in range(len(test_labels)):
    ps = build_pred_set(test_sm_conformal[i], conf_threshold)
    tsz.append(len(ps))
    if CLASS_LABELS[test_labels[i]] in ps: tc += 1
    if len(ps) == 1: ts += 1
n_test_c      = len(test_labels)
test_coverage = tc / n_test_c
test_singleton_rate = ts / n_test_c
test_ambig_rate     = 1 - test_singleton_rate
coverage_ok   = test_coverage >= (1 - alpha)
print(f"\n  [Test Verification]")
print(f"  Coverage:     {test_coverage:.4f}  ({'MET' if coverage_ok else 'NOT MET'})")
print(f"  Singleton:    {ts}/{n_test_c} ({test_singleton_rate:.1%})")
print(f"  Ambiguous:    {n_test_c-ts}/{n_test_c} ({test_ambig_rate:.1%})")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(scores, bins=30, color="#4dabf7", alpha=.7, edgecolor="black")
axes[0].axvline(conf_threshold, color="red", ls="--", lw=2, label=f"Thr={conf_threshold:.3f}")
axes[0].set_title("APS Scores (Cal Set)"); axes[0].legend()
sc2 = Counter(sizes)
su2 = sorted(sc2.keys())
axes[1].bar(su2, [sc2[s] for s in su2], color=["#51cf66","#ffa94d","#ff6b6b"][:len(su2)])
axes[1].set_title("Prediction Set Sizes")
alphas_t = np.linspace(0.01, 0.5, 50)
covs2 = []
for a2 in alphas_t:
    q2 = min(math.ceil((n_cal+1)*(1.-a2))/n_cal, 1.)
    try: t2 = float(np.quantile(scores, q2, method="higher"))
    except TypeError: t2 = float(np.quantile(scores, q2, interpolation="higher"))
    c2 = sum(1 for i in range(n_cal) if CLASS_LABELS[cal_lb_np[i]] in build_pred_set(cal_sm[i], t2))
    covs2.append(c2 / n_cal)
axes[2].plot(alphas_t, covs2, "b-", lw=2, label="Empirical")
axes[2].plot(alphas_t, 1-alphas_t, "r--", alpha=.7, label="1-alpha")
axes[2].axvline(alpha, color="green", ls=":", label=f"alpha={alpha}")
axes[2].set_title("Calibration Curve"); axes[2].legend(); axes[2].grid(True, alpha=.3)
plt.suptitle("Conformal Prediction Diagnostics (V3)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "conformal_diagnostics_v3.png"), dpi=150)
plt.show()


  Conformal Prediction (APS, alpha=0.1)

  Cal samples:  3172
  Temperature:  0.1000  (ConfTS optimized)
  Threshold:    1.0000  (V2 was 1.0 — broken)
  Score mean:   0.9950
  Score std:    0.0288

  [Calibration]
  Coverage:     0.8641  (target>=0.90)
  Singleton:    2191/3172 (69.1%)
  Ambiguous:    981/3172 (30.9%)

  [Test Verification]
  Coverage:     0.8426  (NOT MET)
  Singleton:    1433/2116 (67.7%)
  Ambiguous:    683/2116 (32.3%)


## Cell 14: Export Artifacts

Saves all framework-ready artifacts to `/kaggle/working/`:

| File | Size (approx) | Purpose |
|------|--------------|---------|
| `mini_gat_v3.pt` | ~9 MB | Model weights — copy to `data/models/mini_gat_v4.pt` |
| `mini_gat_v3_best.pt` | ~9 MB | Best val-F1 checkpoint |
| `decision_threshold_v3.json` | < 1 KB | F1-optimal threshold (0.41 in V3) |
| `conformal_calibration_v3.json` | < 5 KB | APS calibration + all test metrics |
| `graph_config_v3.json` | < 10 KB | Model config + node feature spec (framework wiring) |

**Legacy aliases also created** (`conformal_calibration.json`, `graph_config.json`,
`mini_gat.pt`) for backward compatibility with framework default paths.

**Post-download framework integration:**
```
mini_gat_v3.pt          → data/models/mini_gat_v4.pt
conformal_calibration_v3.json → data/models/conformal_calibration.json
graph_config_v3.json    → data/models/graph_config.json
```
Then update `configs/default.yaml`:
```yaml
gnn:
  input_dim: 774
  hidden_dim: 384
  num_gin_layers: 3
  model_path: "data/models/mini_gat_v4.pt"
```


In [15]:
# ============================================================================
# Cell 14: Export All Artifacts
# ============================================================================
import json, torch_geometric, shutil

# 1. Model
model_path = OUTPUT_DIR / "mini_gat_v3.pt"
torch.save(model.cpu().state_dict(), str(model_path))
model.to(device)
print(f"1. Model:      {model_path}  ({model_path.stat().st_size/1024:.0f} KB)")

# 2. Decision threshold (NEW in V3)
thr_export = {
    "decision_threshold": DECISION_THRESHOLD,
    "conformal_temperature": CONFORMAL_TEMPERATURE,
    "val_f1_at_threshold": best_thr_row["f1"],
    "val_precision":       best_thr_row["prec"],
    "val_recall":          best_thr_row["rec"],
    "version": "v3",
    "note": "F1-optimal threshold found by grid search on val set. Use this instead of argmax.",
}
thr_path = OUTPUT_DIR / "decision_threshold_v3.json"
with open(str(thr_path), "w") as f:
    json.dump(thr_export, f, indent=2)
print(f"2. Threshold:  {thr_path}")

# 3. Conformal calibration
cal_export = {
    "alpha": CONFIG["alpha"],
    "threshold": float(conf_threshold),
    "decision_threshold": DECISION_THRESHOLD,
    "conformal_temperature": CONFORMAL_TEMPERATURE,
    "n_calibration": int(n_cal),
    "empirical_coverage": float(empirical_coverage),
    "test_coverage": float(test_coverage),
    "class_names": CLASS_LABELS,
    "singleton_rate": float(singleton_rate),
    "ambiguous_rate": float(ambig_rate),
    "test_singleton_rate": float(test_singleton_rate),
    "test_ambiguous_rate": float(test_ambig_rate),
    "mean_set_size": float(np.mean(sizes)),
    "score_stats": {
        "mean": float(scores.mean()), "std": float(scores.std()),
        "median": float(np.median(scores)), "min": float(scores.min()), "max": float(scores.max()),
    },
    "test_metrics": {
        "accuracy": float(test_metrics["accuracy"]),
        "precision": float(test_metrics["precision"]),
        "recall": float(test_metrics["recall"]),
        "f1": float(test_metrics["f1"]),
        "auc_roc": float(auc_roc),
    },
    "per_language_metrics": {l: {k: float(v) for k,v in m.items()} for l,m in lang_metrics.items()},
    "training_history": {
        "total_epochs": len(history["train_loss"]),
        "best_epoch": int(best_epoch),
        "best_val_f1": float(best_val_f1),
        "best_val_loss": float(best_val_loss),
    },
    "version": "v3",
}
cal_path = OUTPUT_DIR / "conformal_calibration_v3.json"
with open(str(cal_path), "w") as f:
    json.dump(cal_export, f, indent=2)
print(f"3. Calibration:{cal_path}")

# 4. Graph config
gc_export = {
    "model_class": "MiniGINv3",
    "model_config": {
        "input_dim": CONFIG["input_dim"], "hidden_dim": CONFIG["hidden_dim"],
        "num_gin_layers": CONFIG["num_gin_layers"], "dropout": CONFIG["dropout"],
        "num_classes": CONFIG["num_classes"], "embedding_model": CONFIG["embedding_model"],
        "embedding_mode": "mean_pooling",
    },
    "node_features": ["in_degree_norm","out_degree_norm","is_sink","is_source","depth_norm","language_id"],
    "language_ids": CONFIG["language_ids"],
    "sink_patterns": SINK_PATTERNS,
    "source_patterns": SOURCE_PATTERNS,
    "max_nodes": CONFIG["max_nodes"],
    "graph_builder": "tree_sitter+regex" if USE_TREE_SITTER else "regex",
    "torch_version": torch.__version__,
    "torch_geometric_version": torch_geometric.__version__,
    "dataset_info": {
        "total_graphs": len(pyg_dataset),
        "train": len(train_data), "val": len(val_data),
        "cal": len(cal_data), "test": len(test_data),
        "languages": list(CONFIG["language_ids"].keys()),
    },
    "version": "v3",
}
cfg_path = OUTPUT_DIR / "graph_config_v3.json"
with open(str(cfg_path), "w") as f:
    json.dump(gc_export, f, indent=2)
print(f"4. Config:     {cfg_path}")

# Legacy copies
for alias_src, alias_dst in [
    ("conformal_calibration_v3.json", "conformal_calibration.json"),
    ("graph_config_v3.json",          "graph_config.json"),
    ("mini_gat_v3.pt",                "mini_gat.pt"),
]:
    try:
        shutil.copy(str(OUTPUT_DIR / alias_src), str(OUTPUT_DIR / alias_dst))
    except Exception:
        pass

print(f"\nAll artifacts in {OUTPUT_DIR}")


1. Model:      /kaggle/working/mini_gat_v3.pt  (9326 KB)
2. Threshold:  /kaggle/working/decision_threshold_v3.json
3. Calibration:/kaggle/working/conformal_calibration_v3.json
4. Config:     /kaggle/working/graph_config_v3.json

All artifacts in /kaggle/working


## Cell 15: Run Summary Report

Prints a complete summary of the V4 training run:

- **Dataset:** per-source sample counts, per-language split, CWE coverage
- **Training:** best epoch, best val F1, train/val loss at convergence, total time
- **Test results:** F1, Precision, Recall, AUC-ROC, per-CWE top-5
- **Conformal:** threshold, singleton rate, ambiguous rate, empirical coverage
- **Framework instructions:** exact copy commands for artifact deployment

**PhD result framing printed at end:**
> MiniGINv3 trained on ~13,800 real C/C++ vulnerability samples achieves
> F1=X, Precision=Y, Recall=Z on deduplicated held-out test set (BigVul +
> DiverseVul + Devign + PrimeVul). APS conformal prediction (alpha=0.1)
> resolves W% of escalated findings at Stage 2 with guaranteed 90% coverage,
> reducing LLM API calls by W% vs a non-cascade baseline.


In [16]:
# ============================================================================
# Cell 15: Final Summary Report
# ============================================================================
print("="*70)
print("  SEC-C MiniGINv3 — Training Report")
print("="*70)
print(f"\n  MODEL:   MiniGINv3 (3-layer GIN), {pc['trainable']:,} params")
print(f"  INPUT:   {CONFIG['input_dim']} (768 GCB mean-pool + 6 structural)")
print(f"  ARCH:    GIN×3 hidden={CONFIG['hidden_dim']}, residual, BN, dual-pool")
total_v_ds  = sum(1 for s in all_samples if s["label"]==1)
total_s_ds  = len(all_samples) - total_v_ds
print(f"\n  DATA:    {len(pyg_dataset)} graphs ({total_v_ds} vuln, {total_s_ds} safe)")
srcs_used = sorted(set(s.get("source","?") for s in all_samples))
print(f"  SRCS:    {', '.join(srcs_used)}")
print(f"  SPLIT:   {len(train_data)}/{len(val_data)}/{len(cal_data)}/{len(test_data)}")
print(f"\n  TRAIN:   WeightedCE(vuln×{CONFIG['class_weight_vuln']}), smoothing={CONFIG['label_smoothing']}")
print(f"           LR={CONFIG['lr']}, warmup={CONFIG['lr_warmup_epochs']}ep, patience={CONFIG['patience']}")
print(f"  BEST:    epoch {best_epoch}, val_F1={best_val_f1:.4f}")
print(f"\n  THRESHOLD: {DECISION_THRESHOLD:.2f} (F1-optimal on val set, V2 used 0.5)")
print(f"\n  TEST METRICS (threshold={DECISION_THRESHOLD:.2f}):")
print(f"    Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"    Precision: {test_metrics['precision']:.4f}")
print(f"    Recall:    {test_metrics['recall']:.4f}")
print(f"    F1:        {test_metrics['f1']:.4f}")
print(f"    AUC-ROC:   {auc_roc:.4f}")
print(f"\n  PER-LANGUAGE F1:")
for lang, m in lang_metrics.items():
    print(f"    {lang:<15} {m['f1']:.4f}")
print(f"\n  CONFORMAL (alpha={CONFIG['alpha']}):")
print(f"    APS Threshold: {conf_threshold:.4f}")
print(f"    Coverage:      {test_coverage:.4f} (>={1-CONFIG['alpha']:.2f}: {'YES' if coverage_ok else 'NO'})")
print(f"    Singleton:     {test_singleton_rate:.1%}  Ambiguous: {test_ambig_rate:.1%}")
print(f"\n  CONFTS (V5 NEW):\n"
      f"    Temperature: {CONFORMAL_TEMPERATURE:.3f}\n"
      f"\n"
      f"  V4 -> V5 changes:\n"
      f"    Smoothing:  0.1 -> 0.0 (was compressing logit gaps)\n"
      f"    PrimeVul:   Fixed (JSONL direct load)\n"
      f"    VUDENC:     Fixed (list label handling)\n"
      f"    CVEfixes:   Fixed (list label handling)\n"
      f"    ConfTS:     NEW (post-hoc temperature for smaller pred sets)\n"
      f"\n"
      f"  V2 -> V3 changes:")
print(f"    Loss:       FocalLoss(γ=2) -> CE+smooth(0.1)")
print(f"    LR:         1e-3 -> 3e-4 + warmup")
print(f"    Balance:    2.0:1 -> 1.0:1 strict per-language")
print(f"    Arch:       GAT(2L,298K) -> GIN(3L,{pc['trainable']//1000}K) + residual + BN")
print(f"    Pooling:    global_mean -> mean+add concat")
print(f"    Embedding:  CLS -> mean_pool (fixes MISSING pooler warning)")
print(f"    Threshold:  argmax(0.5) -> F1-optimal({DECISION_THRESHOLD:.2f})")
print(f"    Datasets:   +BigVul (CWE-labeled C/C++)")
print(f"    Juliet:     1000 -> {CONFIG['max_juliet_c']} (reduce synthetic bias)")
print(f"="*70)


  SEC-C MiniGINv3 — Training Report

  MODEL:   MiniGINv3 (3-layer GIN), 2,375,046 params
  INPUT:   774 (768 GCB mean-pool + 6 structural)
  ARCH:    GIN×3 hidden=384, residual, BN, dual-pool

  DATA:    21150 graphs (10575 vuln, 10575 safe)
  SRCS:    BigVul, CVEfixes-Py, CrossVul, Devign, DiverseVul, Juliet-C, VUDENC
  SPLIT:   12689/3173/3172/2116

  TRAIN:   WeightedCE(vuln×1.5), smoothing=0.0
           LR=0.0003, warmup=5ep, patience=20
  BEST:    epoch 41, val_F1=0.7623

  THRESHOLD: 0.32 (F1-optimal on val set, V2 used 0.5)

  TEST METRICS (threshold=0.32):
    Accuracy:  0.6999
    Precision: 0.6430
    Recall:    0.8989
    F1:        0.7497
    AUC-ROC:   0.7813

  PER-LANGUAGE F1:
    python          0.8358
    javascript      0.6842
    java            0.6667
    c_cpp           0.7504
    go              0.5714

  CONFORMAL (alpha=0.1):
    APS Threshold: 1.0000
    Coverage:      0.8426 (>=0.90: NO)
    Singleton:     67.7%  Ambiguous: 32.3%

  CONFTS (V5 NEW):
    Temp

## Cell 16: Zip All Artifacts for Single-Click Download

Packages all essential outputs into **`sec_c_gnn_v4_artifacts.zip`** for
download from the Kaggle Output tab.

**Included in zip:**

| File | Description |
|------|-------------|
| `mini_gat_v3.pt` | Model weights (main) |
| `mini_gat_v3_best.pt` | Best val-F1 checkpoint |
| `decision_threshold_v3.json` | F1-optimal decision threshold |
| `conformal_calibration_v3.json` | APS calibration + all metrics |
| `graph_config_v3.json` | Model + graph config (framework wiring) |
| `training_curves_v3.png` | Loss and F1 training curves |
| `evaluation_plots_v3.png` | ROC curve, confusion matrix, per-CWE F1 |
| `conformal_diagnostics_v3.png` | APS score histogram, set sizes, coverage curve |
| `threshold_calibration_v3.png` | F1 vs threshold search plot |
| `eda_overview_v3.png` | Dataset EDA (language/CWE/label distribution) |
| `raw_samples_v3.json` | Balanced sample list (if < 50 MB) |
| `mini_gat_v3_epoch*.pt` | Last 2 periodic epoch checkpoints |

**Excluded:** `pyg_dataset_v3.pt` (large intermediate, not needed for inference).

After download, extract and follow the deployment steps in Cell 14.


In [17]:
# ============================================================================
# Cell 16: Zip all artifacts for single-click download from Kaggle
# ============================================================================
import zipfile, time

_zip_path = OUTPUT_DIR / f"sec_c_gnn_v4_artifacts.zip"

# Files to include (essential artifacts only — large intermediates excluded)
_files = [
    ("mini_gat_v3.pt",                "model weights — load with MiniGINv3"),
    ("mini_gat_v3_best.pt",           "best val-F1 checkpoint"),
    ("decision_threshold_v3.json",    "F1-optimal decision threshold"),
    ("conformal_calibration_v3.json", "APS conformal calibration + all metrics"),
    ("graph_config_v3.json",          "model config + dataset info (framework wiring)"),
    ("eda_overview_v3.png",           "dataset EDA"),
    ("training_curves_v3.png",        "loss / F1 training curves"),
    ("threshold_calibration_v3.png",  "threshold search"),
    ("evaluation_plots_v3.png",       "ROC, confusion matrix, per-CWE F1"),
    ("confts_temperature_v5.png",      "ConfTS temperature search (V5)"),
    ("conformal_diagnostics_v3.png",  "APS score histogram, set sizes, coverage"),
]

# Optionally include raw_samples if small enough (<50 MB)
_rs = OUTPUT_DIR / "raw_samples_v3.json"
if _rs.exists() and _rs.stat().st_size < 50 * 1024 * 1024:
    _files.append(("raw_samples_v3.json", "balanced sample list (for reproducibility)"))

# Include last 2 epoch checkpoints if present
_epoch_ckpts = sorted(OUTPUT_DIR.glob("mini_gat_v3_epoch*.pt"))[-2:]

print(f"Zipping artifacts -> {_zip_path.name}")
print(f"{'File':<42} {'Size':>8}  Description")
print("-" * 75)

with zipfile.ZipFile(str(_zip_path), "w", zipfile.ZIP_DEFLATED) as _zf:
    for _fname, _desc in _files:
        _fp = OUTPUT_DIR / _fname
        if _fp.exists():
            _sz = _fp.stat().st_size
            _zf.write(str(_fp), _fname)
            print(f"  {_fname:<40} {_sz/1024:>7.0f}K  {_desc}")
        else:
            print(f"  {_fname:<40} {'MISSING':>8}  {_desc}")
    for _ck in _epoch_ckpts:
        _zf.write(str(_ck), _ck.name)
        print(f"  {_ck.name:<40} {_ck.stat().st_size/1024:>7.0f}K  epoch checkpoint")

_total_mb = _zip_path.stat().st_size / 1024 / 1024
print(f"\nZip: {_zip_path.name}  ({_total_mb:.1f} MB)")
print(f"Path: {_zip_path}")
print("\nDownload: Kaggle Output tab -> sec_c_gnn_v4_artifacts.zip")


Zipping artifacts -> sec_c_gnn_v4_artifacts.zip
File                                           Size  Description
---------------------------------------------------------------------------
  mini_gat_v3.pt                              9326K  model weights — load with MiniGINv3
  mini_gat_v3_best.pt                         9327K  best val-F1 checkpoint
  decision_threshold_v3.json                     0K  F1-optimal decision threshold
  conformal_calibration_v3.json                  2K  APS conformal calibration + all metrics
  graph_config_v3.json                           2K  model config + dataset info (framework wiring)
  eda_overview_v3.png                          101K  dataset EDA
  training_curves_v3.png                       144K  loss / F1 training curves
  threshold_calibration_v3.png                  97K  threshold search
  evaluation_plots_v3.png                       98K  ROC, confusion matrix, per-CWE F1
  confts_temperature_v5.png                    104K  ConfTS temperatu